# SECA · Motor de inferencia v1
## SheRoomie — Sistema de Evaluación de Candidatas a habitación

**Propósito:** construir la primera versión del motor de inferencia de SECA (encadenamiento hacia adelante) leyendo **directamente** la base de conocimiento `SECA_02_Base_Conocimiento_v1.1.xlsx`.

**Continuidad:** Kumu → Excel → Python *(Sesión 5)* → inferencia *(Sesión 6)* → **motor SECA v1**

| Guía | Qué tomamos tal cual | Qué tuvimos que extender para SECA |
|---|---|---|
| Sesión 5 · Representación | Hechos en un `dict`; reglas como **datos** (`id`, `condiciones`, `conclusion`); `None` = desconocido; trazabilidad con los ID del Excel | Operadores (≠, ≤, entre, pertenece…), escalas ordinales, comparación candidata ↔ casa, varias conclusiones por regla |
| Sesión 6 · Forward chaining | `cumple()` genérico, disparo, ciclo hasta no producir nada nuevo, traza de reglas | Lógica trivalente, estratos, conjunto conflicto, políticas de combinación (resolución de conflictos), cierre de mundo cerrado, explicación y registro |

> **Regla de oro de ambas guías:** el conocimiento vive en el Excel; el motor **no tiene un solo `if` sobre variables del negocio**. Si la experta cambia un umbral, se edita el Excel y se vuelve a ejecutar: el código del motor no se toca. En la sección 13 lo comprobamos automáticamente.

> **Estado del conocimiento:** las 39 reglas están en *Pendiente*. Esta v1 es un **motor de conocimiento candidato**. La v2 se construirá con el conocimiento validado en la entrevista con la experta. Recuerda: el motor puede aplicar *correctamente* una regla *incorrecta*.

**Mapa del notebook**

0. Configuración · 1. Carga del Excel · 2. Contrato de tipos (Excel → Python) · 3. **Auditoría de codificación** · 4. Pesos y políticas de combinación · 5. Compilación de reglas · 6. Validación de la base · 7. Estratificación y flujo · 8. Evaluar condiciones (lógica trivalente) · 9. ¿Por qué no basta el motor de la Sesión 6? · 10. Motor v1 · 11. Caso de punta a punta · 12. Batería de pruebas · 13. Verificación del motor · 14. Brechas y preguntas para la experta · 15. Ruta a la v2

## 0. Configuración

La base de conocimiento vive en la carpeta compartida del equipo en Google Drive:

**Mi unidad › SheRoomie - SBC › `SECA_02_Base_Conocimiento_v1.1.xlsx`**

| Dónde corres el notebook | Qué hace este bloque |
|---|---|
| Colab, con la carpeta en tu *Mi unidad* (la dueña de la carpeta, o quien le agregó un acceso directo) | Monta Drive y lee `/content/drive/MyDrive/SheRoomie - SBC/…` |
| Colab, con la carpeta solo en *Compartido conmigo* | Busca el Excel dentro de la carpeta y lo descarga con tu cuenta (te pide autorizar una vez) |
| Jupyter local | Usa el Excel que esté junto al notebook |

**Recomendado para el equipo:** en Drive, clic derecho sobre la carpeta **SheRoomie - SBC** → **Organizar → Agregar acceso directo** → *Mi unidad*. Así Colab la ve igual que en la cuenta de la dueña y todos leen el mismo archivo.

Cuando subamos una nueva versión del Excel a la misma carpeta, solo hay que cambiar `ARCHIVO_EXCEL`.

In [1]:
import re, sys, copy, ast, inspect, json, random, textwrap, hashlib, unicodedata, operator
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

VERSION_MOTOR = "1.0.0"

# ── Base de conocimiento: carpeta compartida del equipo en Google Drive ───────
# Mi unidad › SheRoomie - SBC › SECA_02_Base_Conocimiento_v1.1.xlsx
ARCHIVO_EXCEL    = "SECA_02_Base_Conocimiento_v1.1.xlsx"
CARPETA_DRIVE    = "SheRoomie - SBC"
ID_CARPETA_DRIVE = "1O2MO6EUjUAqJlJHvhKoj0ojap32pMgaJ"   # para quien la tiene solo en "Compartido conmigo"
RUTA_COLAB       = Path("/content/drive/MyDrive") / CARPETA_DRIVE / ARCHIVO_EXCEL
EN_COLAB         = "google.colab" in sys.modules

def descargar_de_drive():
    # Busca el Excel dentro de la carpeta compartida y lo descarga con la cuenta de quien ejecuta
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    servicio = build("drive", "v3")
    consulta = f"'{ID_CARPETA_DRIVE}' in parents and name = '{ARCHIVO_EXCEL}' and trashed = false"
    encontrados = servicio.files().list(q=consulta, fields="files(id, name)").execute()["files"]
    assert encontrados, (f"No encuentro {ARCHIVO_EXCEL} en la carpeta '{CARPETA_DRIVE}'. "
                         "Pide que te la compartan o revisa el nombre del archivo.")
    destino = Path("/content") / ARCHIVO_EXCEL
    with destino.open("wb") as f:
        descarga, terminado = MediaIoBaseDownload(f, servicio.files().get_media(fileId=encontrados[0]["id"])), False
        while not terminado:
            _, terminado = descarga.next_chunk()
    return destino

def ubicar_excel():
    if not EN_COLAB:                       # Jupyter local: el Excel junto al notebook
        return Path(ARCHIVO_EXCEL)
    from google.colab import drive
    drive.mount("/content/drive")
    if RUTA_COLAB.exists():                # la carpeta está en tu Mi unidad (o tienes el acceso directo)
        return RUTA_COLAB
    print(f"'{CARPETA_DRIVE}' no está en tu Mi unidad; descargo el Excel desde la carpeta compartida…")
    return descargar_de_drive()

RUTA_EXCEL = ubicar_excel()
assert RUTA_EXCEL.exists(), (
    f"No encuentro {RUTA_EXCEL}. En Colab: revisa que tengas acceso a la carpeta '{CARPETA_DRIVE}'. "
    "En Jupyter local: coloca el Excel junto a este notebook."
)
print(f"Leyendo: {RUTA_EXCEL}")
HASH_BC = hashlib.sha256(RUTA_EXCEL.read_bytes()).hexdigest()[:12]
print(f"Motor SECA v{VERSION_MOTOR} · Base de conocimiento: {RUTA_EXCEL.name} · sha256 {HASH_BC}")

Leyendo: SECA_02_Base_Conocimiento_v1.1.xlsx
Motor SECA v1.0.0 · Base de conocimiento: SECA_02_Base_Conocimiento_v1.1.xlsx · sha256 993d8cd41148


## 1. Carga de la base de conocimiento

Leemos las tres hojas tal como las llenó el equipo. Las tres primeras filas de cada hoja son título, descripción y separador; los encabezados están en la fila 4.

In [2]:
def leer_hoja(nombre):
    df = pd.read_excel(RUTA_EXCEL, sheet_name=nombre, header=3, dtype=str)
    return (df.dropna(subset=["ID"]).fillna("")
              .apply(lambda col: col.str.strip()).reset_index(drop=True))

xl_dic    = leer_hoja("Diccionario")
xl_reglas = leer_hoja("Reglas")
xl_exc    = leer_hoja("Excepciones")

PREFIJOS = {"V": "Variables (hechos)", "C": "Conclusiones intermedias", "A": "Acciones", "D": "Decisiones"}
inventario = xl_dic["ID"].str[0].map(PREFIJOS).value_counts().reindex(PREFIJOS.values()).to_frame("elementos")
inventario.loc["Reglas"] = len(xl_reglas)
inventario.loc["Excepciones / pendientes"] = len(xl_exc)
inventario

,elementos
ID,
Variables (hechos),35
Conclusiones intermedias,9
Acciones,10
Decisiones,3
Reglas,39
Excepciones / pendientes,25


In [3]:
# ¿De dónde viene el conocimiento y qué tan validado está?
pd.crosstab(xl_reglas["Origen"], xl_reglas["Situación en la app"], margins=True, margins_name="Total")

Situación en la app,Implementado,No existe,Parcial,Total
Origen,,,,
Datos / sistema existente,6,0,0,6
IA generativa,0,18,0,18
Supuesto del equipo,0,11,4,15
Total,6,29,4,39


Todas las reglas están en estado **Pendiente** y más de la mitad viene de *IA generativa* o *Supuesto del equipo*. Por eso cada regla compilada conservará su **origen, fuente de validación y estado**: la explicación de un dictamen siempre dirá con qué tipo de conocimiento se obtuvo.

## 2. Contrato de tipos: cómo pasa cada dato del Excel a Python

En la Sesión 5 vimos `True`, `False` y `None`. Con el Excel de SECA eso no alcanza: hay booleanos escritos de dos formas, variables de tres valores que *parecen* booleanas, escalas con orden, números con unidad y acciones que se acumulan.

**Contrato v1:**

| Tipo | En el Excel | En Python | Ejemplo |
|---|---|---|---|
| booleano | `sí; no` o `true; false` | `bool` | `'sí'` → `True`, `'false'` → `False` |
| ordinal | catálogo con orden | `str` + **rango** entero para comparar | `relajada=0 < ordenada=1 < muy_ordenada=2` |
| numérico | `numérico (MXN / % / meses)`, `0..n` | `int` / `float` en la unidad declarada | `'33%'` → `33` (no `0.33`) |
| categórico | catálogo sin orden | `str` canónica (ASCII, snake_case) | `'solo_afuera'` |
| acción | `A01…A10` | elemento del conjunto `acciones` | `frozenset({'generar_explicacion', …})` |
| desconocido | celda vacía, `null`, `vacío` | `None` | **nunca** se interpreta como `False` |

Primero, la forma canónica de cualquier literal: sin acentos, minúsculas y snake_case. Así `'sí'` y `'si'` son el mismo valor.

In [4]:
def canon(texto):
    ascii_ = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", "_", ascii_.strip().lower())

def tokens(texto):
    # 'unverified; pending; verified' → ['unverified', 'pending', 'verified']
    return [canon(t) for t in str(texto).split(";") if canon(t)]

def num(texto):
    # '33%' → 33 · '+1' → 1 · '45.5' → 45.5
    x = float(str(texto).replace("%", "").strip())
    return int(x) if x.is_integer() else x

canon("Sí"), canon("Muy ordenada"), tokens("sí; parcial; no"), num("33%"), num("+2")

('si', 'muy_ordenada', ['si', 'parcial', 'no'], 33, 2)

### 2.1 Escalas ordinales

El Excel **no codifica el orden** de sus escalas (solo V22 lo menciona en la descripción). Sin orden, `'relajada' < 'muy_ordenada'` compararía texto alfabéticamente y daría resultados absurdos. Declaramos las escalas como **conocimiento** (de menor a mayor) y el motor compara por la posición.

In [5]:
ESCALAS = {
    "nivel_limpieza":          ["relajada", "ordenada", "muy_ordenada"],  # V22 · orden en su descripción
    "casa_limpieza_esperada":  ["relajada", "ordenada", "muy_ordenada"],  # V27 · misma escala que V22
    "frecuencia_visitas":      ["nunca", "ocasional", "frecuente"],       # V21 · implícito en R20 ('es mayor a')
    "casa_tolerancia_visitas": ["nunca", "ocasional", "frecuente"],       # V28 · misma escala que V21
    "referencias_verificadas": ["no", "parcial", "si"],                   # V04 · ternaria, NO booleana
    "nivel_solvencia":         ["baja", "media", "alta"],                 # C04
    "nivel_compatibilidad":    ["baja", "media", "alta"],                 # C08
    "dictamen": ["aprobada", "aprobada_con_condicion",                    # D01 · orden de severidad (E02)
                 "revision_manual", "rechazada_para_habitacion"],
}

# Clave de comparación: valor → posición en su escala (None si el valor es desconocido)
CLAVES = {var: {v: i for i, v in enumerate(esc)}.get for var, esc in ESCALAS.items()}
identidad = lambda v: v

CLAVES["nivel_limpieza"]("relajada") < CLAVES["nivel_limpieza"]("muy_ordenada")

True

### 2.2 Inferencia del tipo de cada elemento del Diccionario

El tipo se deduce de la columna *Valores posibles* con una **tabla de predicados**: gana el primero que se cumple. Es el mismo principio del motor: la decisión está en una tabla de datos, no en una cadena de `if`.

In [6]:
LITERALES_BOOL = {"si": True, "true": True, "no": False, "false": False}

REGLAS_DE_TIPO = [
    ("excluida",   lambda f: f["situacion"] == "Excluida"),
    ("accion",     lambda f: f["id"].startswith("A")),
    ("ordinal",    lambda f: f["nombre"] in ESCALAS),
    ("booleano",   lambda f: set(f["tokens"]) - {"null"} in ({"si", "no"}, {"true", "false"})),
    ("numerico",   lambda f: bool(re.search(r"num[eé]rico|entero|0\.\.n|\d\s*[–-]\s*\d", f["valores"]))
                             or all(re.fullmatch(r"\d+\+?", t) for t in f["tokens"])),
    ("texto",      lambda f: "texto libre" in f["valores"]),
    ("categorico", lambda f: True),
]

def ficha(fila):
    f = {"id": fila["ID"], "nombre": fila["Concepto / variable"], "categoria": fila["Tipo"],
         "valores": fila["Valores posibles"], "tokens": tokens(fila["Valores posibles"]),
         "descripcion": fila["Descripción"], "situacion": fila["Situación en la app"],
         "origen": fila["Origen"], "estado": fila["Estado de validación"]}
    f["tipo"] = next(tipo for tipo, predicado in REGLAS_DE_TIPO if predicado(f))
    return f

CATALOGO = {f["nombre"]: f for f in map(ficha, xl_dic.to_dict("records"))}

# Variables que el Excel NO declara, pero que la normalización v1 necesita (propuestas para el Excel v1.2)
EXTENSIONES_V1 = {
    "acciones":           {"tipo": "conjunto", "motivo": "R05, R34–R36 concluyen 'accion' y R03/R37 la añaden entre paréntesis: es un conjunto, no un valor único."},
    "decision_permitida": {"tipo": "booleano", "motivo": "R39 concluye un código de error de la API ('no permitida (BAD_REQUEST)')."},
    "brecha_limpieza":    {"tipo": "numerico", "motivo": "R18/R19 usan 'diferencia de niveles', que no está declarada."},
}
VARIABLES = {**CATALOGO, **{n: {"nombre": n, "id": "ext", "tokens": [], **e} for n, e in EXTENSIONES_V1.items()}}
EXCLUIDAS = {n for n, f in CATALOGO.items() if f["tipo"] == "excluida"}
ACCIONES_DECLARADAS = {n for n, f in CATALOGO.items() if f["tipo"] == "accion"}

pd.Series({n: f["tipo"] for n, f in CATALOGO.items()}).value_counts().to_frame("elementos")

,elementos
categorico,13
booleano,13
numerico,10
accion,10
ordinal,8
excluida,2
texto,1


### 2.3 Conversión de literales y tabla del contrato

`a_python(variable, literal)` convierte cualquier valor escrito en el Excel al tipo Python de su variable. También hay un validador por tipo que usaremos como "aduana" para los hechos de entrada.

In [7]:
CONVERTIR = {
    "booleano":   lambda lit: LITERALES_BOOL[canon(lit)],
    "ordinal":    canon,
    "categorico": canon,
    "numerico":   num,
    "conjunto":   canon,   # cada elemento del conjunto 'acciones'
    "accion":     canon,
}

def a_python(variable, literal):
    return CONVERTIR[VARIABLES[variable]["tipo"]](literal)

VALIDA = {
    "booleano":   lambda v, f: isinstance(v, bool),
    "ordinal":    lambda v, f: v in ESCALAS[f["nombre"]],
    "categorico": lambda v, f: v in f["tokens"],
    "numerico":   lambda v, f: isinstance(v, (int, float)) and not isinstance(v, bool),
    "conjunto":   lambda v, f: isinstance(v, (set, frozenset)),
    "texto":      lambda v, f: isinstance(v, str),
    "excluida":   lambda v, f: True,    # se acepta en la entrada, pero se descarta antes de razonar
    "accion":     lambda v, f: False,   # una acción es una salida, nunca un hecho de entrada
}

REPRESENTACION = {
    "booleano":   ("bool", "True / False · None = desconocido"),
    "ordinal":    ("str + rango", None),
    "numerico":   ("int | float", "unidad declarada (MXN, %, meses) · None = desconocido"),
    "categorico": ("str", "valor canónico del catálogo"),
    "texto":      ("—", "texto libre: no entra a reglas"),
    "excluida":   ("—", "PROHIBIDA en reglas (no discriminación)"),
    "accion":     ("str ∈ acciones", "elemento del conjunto `acciones`"),
    "conjunto":   ("frozenset[str]", "acumula acciones"),
}

contrato = pd.DataFrame([{
    "ID": f["id"], "variable": n, "valores en Excel": f["valores"], "tipo": f["tipo"],
    "Python": REPRESENTACION[f["tipo"]][0],
    "detalle": " < ".join(ESCALAS.get(n, [])) or REPRESENTACION[f["tipo"]][1],
} for n, f in CATALOGO.items()])
contrato

,ID,variable,valores en Excel,tipo,Python,detalle
0,V01,estado_verificacion_identidad,unverified; pending; verified; rejected,categorico,str,valor canónico del catálogo
1,V02,telefono_verificado,true; false,booleano,bool,True / False · None = desconocido
2,V03,cuenta_activa,true; false,booleano,bool,True / False · None = desconocido
3,V04,referencias_verificadas,sí; parcial; no,ordinal,str + rango,no < parcial < si
4,V05,metodo_pago_validado,sí; no,booleano,bool,True / False · None = desconocido
5,V06,ingreso_mensual_comprobable,numérico (MXN),numerico,int | float,"unidad declarada (MXN, %, meses) · None = desconocido"
6,V07,renta_mensual_referencia,numérico (MXN),numerico,int | float,"unidad declarada (MXN, %, meses) · None = desconocido"
7,V08,ratio_renta_ingreso,numérico (%),numerico,int | float,"unidad declarada (MXN, %, meses) · None = desconocido"
8,V09,tipo_ingreso,nomina; honorarios; negocio_propio; apoyo_familiar_beca; sin_ingreso_comprobable,categorico,str,valor canónico del catálogo
9,V10,antiguedad_laboral_meses,numérico (meses),numerico,int | float,"unidad declarada (MXN, %, meses) · None = desconocido"


## 3. Auditoría de codificación del Excel

Antes de compilar, revisamos automáticamente el Excel buscando valores y salidas que un programa no puede interpretar sin ambigüedad. Cada hallazgo dice **qué está mal**, **qué tan grave es** y **cómo lo codifica la v1** (para que en el Excel v1.2 se corrija en origen).

Severidad: **Crítico** (el motor no puede razonar o da resultados incorrectos) · **Alto** · **Medio** · **Bajo** · **Info**.

In [8]:
OPERADOR_TEXTO = {
    "es igual a": "eq", "es diferente de": "ne",
    "es menor a": "lt", "es menor o igual a": "le",
    "es mayor a": "gt", "es mayor o igual a": "ge",
    "entre": "entre", "pertenece a": "en", "no pertenece a": "no_en",
}
ALIAS_VARIABLES = {"accion": "acciones"}
SEVERIDAD = {"Crítico": 0, "Alto": 1, "Medio": 2, "Bajo": 3, "Info": 4}

hallazgos = []
def hallazgo(elemento, categoria, severidad, detalle, codificacion):
    hallazgos.append({"elemento": elemento, "categoría": categoria, "severidad": severidad,
                      "hallazgo": detalle, "cómo se codifica en v1": codificacion})

def condiciones_crudas(fila):
    pares = [(fila["Condición 1"], fila["Operador"], fila["Valor 1"]),
             (fila["Condición 2"], fila["Operador2"], fila["Valor 2"])]
    return [(canon(v), o.strip().lower(), x) for v, o, x in pares if v]

def conclusiones_crudas(fila):
    # 'estado_solicitud (+ acción notificar_x)' → [('estado_solicitud','pending'), ('acciones','notificar_x')]
    texto, valor = fila["Conclusión"], fila["Valor conclusión"]
    principal = canon(re.split(r"\s*\(", texto)[0])
    principal = ALIAS_VARIABLES.get(principal, principal)
    extra = re.findall(r"\(\+\s*acci[oó]n\s+(\w+)\)", texto)
    valores = re.split(r"\s\+\s", valor) if principal == "acciones" else [valor]
    return [(principal, canon(v)) for v in valores] + [("acciones", canon(a)) for a in extra]

filas = xl_reglas.to_dict("records")
ids_de = lambda nombres: ", ".join(CATALOGO[n]["id"] for n in nombres)

### 3.1 Diccionario: booleanos, ternarias, escalas y variables sin uso

In [9]:
# Booleanos escritos de dos formas
bool_tf = [n for n, f in CATALOGO.items() if f["tipo"] == "booleano" and "true" in f["tokens"]]
bool_sn = [n for n, f in CATALOGO.items() if f["tipo"] == "booleano" and "si" in f["tokens"]]
hallazgo(f"{ids_de(bool_tf)} | {ids_de(bool_sn)}", "Booleanos", "Medio",
         f"Dos codificaciones de booleano: true/false en {len(bool_tf)} campos de BD y sí/no en {len(bool_sn)} conceptos SECA. "
         "Las reglas mezclan ambas (R14 usa 'true'/'false'; R12, R21, R24 usan 'sí'/'no').",
         "Ambas → bool. En reglas y entradas: 'sí'/'si'/'true' → True · 'no'/'false' → False.")

# 'si' sin acento
sin_acento = [n for n, f in CATALOGO.items() if re.search(r"(^|;\s*)si(\s*;|$)", f["valores"])]
hallazgo(ids_de(sin_acento), "Literales", "Bajo",
         "Se escribe 'si' (sin acento) en " + ids_de(sin_acento) + " y 'sí' en el resto. En texto son valores distintos.",
         "canon() quita acentos: 'sí' y 'si' → 'si'.")

# Variables de 3 valores o con null que parecen booleanas
ternarias = [n for n, f in CATALOGO.items()
             if {"si", "no"} <= set(f["tokens"]) and f["tipo"] != "booleano"]
con_null = [n for n, f in CATALOGO.items() if "null" in f["tokens"] and f["tipo"] != "excluida"]
hallazgo(ids_de(ternarias + con_null), "Booleanos", "Medio",
         "Parecen booleanas pero no lo son: " + ", ".join(f"{CATALOGO[n]['id']} ({CATALOGO[n]['valores']})" for n in ternarias)
         + ". " + ids_de(con_null) + " admite null.",
         "V04 → ordinal no < parcial < si · V20/V25 → categórica str · null → None (desconocido), nunca False.")

# Escalas sin orden declarado en el Excel
sin_orden = [n for n in ESCALAS if n in CATALOGO and not re.search(r"<|orden:", CATALOGO[n]["descripcion"], re.I)]
hallazgo(ids_de(sin_orden), "Escalas", "Medio",
         "Escalas ordinales sin orden declarado. El orden de D01 (severidad) solo aparece como texto en E02.",
         "ESCALAS (sección 2.1): cada valor tiene un rango entero. Proponer columna 'Escala' en el Excel v1.2.")
dominio_vs_escala = [n for n in ESCALAS if n in CATALOGO and set(ESCALAS[n]) != set(CATALOGO[n]["tokens"])]
hallazgo(ids_de(dominio_vs_escala) or f"{len(ESCALAS)} escalas", "Escalas", "Info",
         f"Escalas que no coinciden con 'Valores posibles': {len(dominio_vs_escala)}.",
         "Las escalas usan exactamente los valores del Diccionario.")

# Numéricos con centinelas o etiquetas
centinelas = [n for n, f in CATALOGO.items() if f["tipo"] == "numerico"
              and any(re.fullmatch(r"\d+\+", t) or t.startswith("sin_") for t in f["tokens"])]
hallazgo(ids_de(centinelas), "Numéricos", "Bajo",
         "Numéricos mezclados con etiquetas: " + "; ".join(f"{CATALOGO[n]['id']} = '{CATALOGO[n]['valores']}'" for n in centinelas) + ".",
         "diferencias_blandas → int ≥ 0 ('3+' es una etiqueta de reporte, no un valor). 'sin_resenas' → None + conteo aparte.")

In [10]:
# Variables del Diccionario que ninguna regla usa
usadas = {v for fila in filas for v, _, x in condiciones_crudas(fila)}
usadas |= {canon(x) for fila in filas for _, _, x in condiciones_crudas(fila) if canon(x) in CATALOGO}
usadas |= {v for fila in filas for v, _ in conclusiones_crudas(fila)}
por_id = {f["id"]: n for n, f in CATALOGO.items()}
usadas |= {por_id[i] for i in re.findall(r"V\d\d", CATALOGO["ratio_renta_ingreso"]["descripcion"])}  # fórmula de V08

sin_uso = [n for n, f in CATALOGO.items()
           if f["categoria"] == "Hecho / Variable" and n not in usadas and f["tipo"] not in ("excluida", "texto")]
hallazgo(ids_de(sin_uso), "Cobertura", "Medio",
         "Variables declaradas que ninguna regla usa: " + ", ".join(sin_uso)
         + ". Ojo con V03 cuenta_activa: una cuenta suspendida puede recibir dictamen.",
         "Se aceptan como entrada, pero no influyen. Decidir con la experta si entran a reglas o se retiran.")

# Doble rol: entrada y conclusión a la vez
doble_rol = [n for n, f in CATALOGO.items() if f["categoria"] == "Hecho / Variable"
             and any(v == n for fila in filas for v, _ in conclusiones_crudas(fila))]
hallazgo(ids_de(doble_rol), "Salidas", "Bajo",
         "Variable de entrada que también es conclusión: " + ", ".join(doble_rol) + " (V32 = bookings.status y R03 lo concluye).",
         "Los hechos de entrada prevalecen (setdefault). Si el motor concluye otro valor, queda registrado como discrepancia.")

hallazgo("C09", "Pesos", "Info",
         "C09 trae pesos 25/20/20/15/10/10 (incluye 10 pts por tener hijos, variable excluida). Es candidata ↔ candidata.",
         "SECA NO importa esos pesos. Los únicos pesos de la v1 son los de diferencias_blandas (R17–R20) y la severidad de D01 (E02).")

### 3.2 Reglas: valores que no son valores

In [11]:
PATRONES_VALOR = [
    # (regex sobre el valor, severidad, hallazgo, codificación v1)
    (r"^vac[ií]o$", "Medio",
     "'vacío' usado como valor. Si se compara como texto, 'vacío' ≠ None y la regla nunca se cumple.",
     "'es igual a vacío' → operador es_desconocido · 'es diferente de vacío' → es_conocido."),
    (r"%", "Medio",
     "Umbrales escritos como texto con '%'.",
     "num('33%') → 33. V08 está en porcentaje (V07 ÷ V06 × 100), no en fracción 0–1."),
    (r"^\d+%? y \d+%?$", "Medio",
     "'entre 33% y 45%' no dice si incluye los extremos. Leído inclusivo, en ratio = 33 % se cumplen R06/R07 (≤ 33) y R08 a la vez: alta y media.",
     "Intervalo semiabierto (33, 45]: R06/R07 cubren ≤ 33, R08 (33, 45], R09 > 45. Sin huecos ni solapes."),
    (r"^\+\d+$", "Crítico",
     "'+1' / '+2' son incrementos, no valores. Con asignación, cada regla BORRA a la anterior (dos diferencias de +1 dan 1, no 2) y nadie produce 0.",
     "Política 'suma' sobre diferencias_blandas, con valor 0 por cierre. Los pesos (+1/+2) se respetan."),
    (r"como m[aá]ximo", "Alto",
     "'media como máximo (no puede ser alta)' es un tope escrito en lenguaje natural, no un valor.",
     "Se extrae 'media' y nivel_solvencia usa política 'mínimo': min(alta, media) = media; min(baja, media) = baja."),
    (r"^\{.*\}$", "Medio",
     "Un conjunto de variables usado como valor y 'flexible' usado como variable.",
     "Se reescribe como 3 condiciones atómicas (normalización de R16)."),
    (r"^(true|false)$", "Bajo",
     "Literales true/false dentro de reglas (el resto usa sí/no).",
     "Ambos → bool."),
    (r";", "Bajo",
     "Lista de valores dentro de una celda ('alta; media').",
     "frozenset({'alta', 'media'}) con operador 'en'."),
]

for regex, severidad, detalle, codif in PATRONES_VALOR:
    reglas_con = sorted({f["ID"] for f in filas
                         for x in [f["Valor 1"], f["Valor 2"], f["Valor conclusión"]] if x and re.search(regex, x.strip(), re.I)})
    reglas_con and hallazgo(", ".join(reglas_con), "Valores", severidad, detalle, codif)

# Variables no declaradas usadas en condiciones
no_decl = sorted({(f["ID"], v) for f in filas for v, _, _ in condiciones_crudas(f) if v not in CATALOGO})
hallazgo(", ".join(sorted({i for i, _ in no_decl})), "Integridad", "Alto",
         "Condiciones sobre variables no declaradas: " + "; ".join(f"{i}: '{v}'" for i, v in no_decl) + ".",
         "R16 → 3 condiciones atómicas · R18/R19 → variable derivada brecha_limpieza = rango(casa) − rango(candidata).")

# Comparaciones entre variables (candidata ↔ casa)
refs = sorted({f["ID"] for f in filas for _, _, x in condiciones_crudas(f) if canon(x) in CATALOGO})
hallazgo(", ".join(refs), "Valores", "Info",
         "Comparaciones variable contra variable (candidata ↔ casa). Es correcto, pero el parser debe distinguirlas de un literal.",
         "Ref('casa_…'): se resuelve contra la memoria de trabajo y se compara con la misma escala.")

# Conector O
disy = [f["ID"] for f in filas if f["Conector"] == "O"]
hallazgo(", ".join(disy), "Conectores", "Medio",
         "Disyunción (O) en el antecedente. El cumple() de la Sesión 6 solo sabe de Y.",
         "Se divide en dos reglas equivalentes (R04a, R04b, R12a, R12b) con la misma conclusión.")

# Conclusiones compuestas o no declaradas
compuestas = [f["ID"] for f in filas if "(+" in f["Conclusión"] or re.search(r"\s\+\s", f["Valor conclusión"])]
hallazgo(", ".join(compuestas), "Salidas", "Alto",
         "Dos conclusiones en una sola celda ('estado_solicitud (+ acción …)', 'generar_explicacion + registrar_dictamen').",
         "Se separan en una lista de conclusiones: [('estado_solicitud','pending'), ('acciones','notificar_…')].")
accion_multi = [f["ID"] for f in filas if canon(re.split(r"\s*\(", f["Conclusión"])[0]) == "accion"]
hallazgo(", ".join(accion_multi), "Salidas", "Alto",
         f"'accion' no está en el Diccionario y {len(accion_multi)} reglas le asignan valores distintos: con asignación simple la última borra a las demás.",
         "Variable 'acciones' de tipo conjunto: cada regla agrega su acción; ninguna borra.")
raras = [f["ID"] for f in filas if not re.fullmatch(r"[a-z_]+( \(\+ acci[oó]n \w+\))?", f["Conclusión"])]
hallazgo(", ".join(raras), "Salidas", "Medio",
         "Conclusión que no es una variable: " + "; ".join(f"{f['ID']}: '{f['Conclusión']} = {f['Valor conclusión']}'" for f in filas if f["ID"] in raras) + ".",
         "R39 → decision_permitida = False (es un error de la API, no un valor de la base).")

### 3.3 Salidas inalcanzables (el hallazgo más importante)

Una regla solo puede dispararse si alguien produce los valores que exige. Para las **entradas** (variables V) asumimos que pueden llegar con cualquier valor; para las **conclusiones** (C, D), el valor tiene que salir de alguna regla alcanzable. Calculamos el punto fijo leyendo el Excel **literalmente**.

In [12]:
ENTRADAS = {n for n, f in CATALOGO.items() if f["categoria"] == "Hecho / Variable"}

def satisfacible(var, op, valor, producidos):
    esperado = canon(valor)
    return {
        True:  lambda: True,                                               # entrada: puede tomar cualquier valor
        False: lambda: {"es igual a": esperado in producidos[var],
                        "pertenece a": bool(set(tokens(valor)) & producidos[var])
                        }.get(op, bool(producidos[var])),
    }[var in ENTRADAS]()

def alcanzabilidad_literal(filas):
    producidos, alcanzables, cambio = defaultdict(set), set(), True
    while cambio:
        nuevas = {f["ID"] for f in filas if f["ID"] not in alcanzables and
                  (any if f["Conector"] == "O" else all)(satisfacible(v, o, x, producidos) for v, o, x in condiciones_crudas(f))}
        for f in (f for f in filas if f["ID"] in nuevas):
            for var, val in conclusiones_crudas(f):
                producidos[var].add(val)
        alcanzables |= nuevas
        cambio = bool(nuevas)
    return alcanzables, producidos

alcanzables, producidos = alcanzabilidad_literal(filas)

def motivo(fila):
    fallas = [(v, o, x) for v, o, x in condiciones_crudas(fila) if not satisfacible(v, o, x, producidos)]
    return "; ".join(
        f"'{v}' no está declarada" if v not in CATALOGO else
        f"exige {v} {o} '{x}', pero solo se produce {sorted(producidos[v]) or '∅'}"
        for v, o, x in fallas)

inalcanzables = pd.DataFrame([{"regla": f["ID"], "concluye": f"{f['Conclusión']} = {f['Valor conclusión']}", "por qué nunca se activa": motivo(f)}
                              for f in filas if f["ID"] not in alcanzables])
print(f"Reglas que NUNCA pueden activarse leyendo el Excel literalmente: {len(inalcanzables)} de {len(filas)}")
inalcanzables

Reglas que NUNCA pueden activarse leyendo el Excel literalmente: 12 de 39


,regla,concluye,por qué nunca se activa
0,R03,estado_solicitud (+ acción notificar_anfitriona_nueva_solicitud) = pending,"exige puede_solicitar es igual a 'sí', pero solo se produce ['no']; exige propiedad_solicitable es igual a 'sí', pero solo se produce ['no']"
1,R16,restriccion_dura_violada = sí,'flexible' no está declarada
2,R18,diferencias_blandas = +1,'diferencia_de_niveles' no está declarada
3,R19,diferencias_blandas = +2,'diferencia_de_niveles' no está declarada
4,R21,nivel_compatibilidad = alta,"exige restriccion_dura_violada es igual a 'no', pero solo se produce ['si']; exige diferencias_blandas es igual a '0', pero solo se produce ['+1']"
5,R22,nivel_compatibilidad = media,"exige restriccion_dura_violada es igual a 'no', pero solo se produce ['si']; exige diferencias_blandas es igual a '1', pero solo se produce ['+1']"
6,R23,nivel_compatibilidad = baja,"exige restriccion_dura_violada es igual a 'no', pero solo se produce ['si']"
7,R25,dictamen = aprobada,"exige nivel_compatibilidad es igual a 'alta', pero solo se produce ∅"
8,R26,dictamen = aprobada,"exige nivel_compatibilidad es igual a 'media', pero solo se produce ∅"
9,R27,dictamen = aprobada_con_condicion,"exige nivel_compatibilidad pertenece a 'alta; media', pero solo se produce ∅"


In [13]:
# ¿Qué salidas declaradas nunca se producen?
salidas_declaradas = {n: set(f["tokens"]) for n, f in CATALOGO.items() if f["categoria"] in ("Conclusión intermedia", "Conclusión / Decisión")}
nunca = {n: sorted(vals - producidos[n]) for n, vals in salidas_declaradas.items() if vals - producidos[n] and n != "puntaje_compatibilidad_roomies"}
acciones_nunca = sorted(ACCIONES_DECLARADAS - producidos["acciones"])

pd.DataFrame([{"salida": f"{CATALOGO[n]['id']} {n}", "valores que ninguna regla alcanzable produce": ", ".join(v)} for n, v in nunca.items()]
             + [{"salida": "A · acciones", "valores que ninguna regla alcanzable produce": ", ".join(acciones_nunca)}])

,salida,valores que ninguna regla alcanzable produce
0,C01 puede_solicitar,si
1,C02 propiedad_solicitable,si
2,C03 expediente_completo,si
3,C05 compensacion_disponible,no
4,C06 restriccion_dura_violada,no
5,C07 diferencias_blandas,"0, 1, 2, 3+"
6,C08 nivel_compatibilidad,"alta, baja, media"
7,D01 dictamen,aprobada
8,D02 condicion_aplicada,"meses_adelantados, ninguna"
9,D03 estado_solicitud_resultante,pending


In [14]:
for var, cuales in [("puede_solicitar", "R03"), ("propiedad_solicitable", "R03"),
                    ("restriccion_dura_violada", "R21–R23"), ("compensacion_disponible", "R29")]:
    contrario = "no" if "si" in producidos[var] else "sí"
    verbo = "exigen" if "–" in cuales else "exige"
    hallazgo(f"{CATALOGO[var]['id']} / {cuales}", "Salidas inalcanzables", "Crítico",
             f"Las reglas solo producen {var} = {sorted(producidos[var])}; {cuales} {verbo} '{contrario}', que ninguna regla produce.",
             f"Cierre de mundo cerrado condicionado: si todas las reglas que concluyen {var} pudieron evaluarse y ninguna se disparó, "
             f"se publica {var} = {contrario}. Si alguna quedó indeterminada por falta de datos, queda desconocido.")
hallazgo("D01 aprobada", "Salidas inalcanzables", "Crítico",
         "Como consecuencia, nivel_compatibilidad nunca se deriva y el dictamen 'aprobada' es INALCANZABLE: con el Excel literal, SECA nunca aprueba a nadie.",
         "Se resuelve con los cierres anteriores (sin inventar reglas nuevas).")
hallazgo("A03, A04, A05 · D02", "Salidas huérfanas", "Alto",
         "condicion_aplicada = aval / deposito_ampliado se concluye, pero ninguna regla dispara solicitar_aval, solicitar_deposito_ampliado "
         "ni solicitar_meses_adelantados. 'meses_adelantados' nunca se elige aunque R13 lo acepte como compensación (E09).",
         "Se documenta como brecha para la experta. La v1 no inventa esas reglas.")

### 3.4 Frontera de umbrales (ratio renta/ingreso)

In [15]:
# Lectura literal del Excel (extremos inclusivos) vs. codificación v1 (33, 45]
literal = {"R06/R07 (≤ 33)": lambda x: x <= 33, "R08 literal [33, 45]": lambda x: 33 <= x <= 45,
           "R08 v1 (33, 45]": lambda x: 33 < x <= 45, "R09 (> 45)": lambda x: x > 45}
pd.DataFrame({f"ratio = {x} %": {k: f(x) for k, f in literal.items()} for x in (32.9, 33, 33.1, 45, 45.1)})

,ratio = 32.9 %,ratio = 33 %,ratio = 33.1 %,ratio = 45 %,ratio = 45.1 %
R06/R07 (≤ 33),True,True,False,False,False
"R08 literal [33, 45]",False,True,True,True,False
"R08 v1 (33, 45]",False,False,True,True,False
R09 (> 45),False,False,False,False,True


In [16]:
hallazgo(f"{len(filas)} reglas", "Metadatos", "Info",
         f"{(xl_reglas['Estado'] == 'Pendiente').sum()}/{len(filas)} reglas en Pendiente; "
         f"{xl_reglas['Origen'].isin(['IA generativa', 'Supuesto del equipo']).sum()} de origen IA generativa o Supuesto del equipo.",
         "Cada regla compilada conserva origen, fuente y estado. La v2 podrá filtrar por Estado = Validado.")

auditoria = (pd.DataFrame(hallazgos)
             .sort_values("severidad", key=lambda s: s.map(SEVERIDAD), kind="stable").reset_index(drop=True))
auditoria.index += 1
print(auditoria["severidad"].value_counts().reindex(SEVERIDAD).dropna().astype(int).to_string())
auditoria

severidad
Crítico     6
Alto        5
Medio      10
Bajo        5
Info        4


,elemento,categoría,severidad,hallazgo,cómo se codifica en v1
1,"R17, R18, R19, R20",Valores,Crítico,"'+1' / '+2' son incrementos, no valores. Con asignación, cada regla BORRA a la anterior (dos diferencias de +1 dan 1, no 2) y nadie produce 0.","Política 'suma' sobre diferencias_blandas, con valor 0 por cierre. Los pesos (+1/+2) se respetan."
2,C01 / R03,Salidas inalcanzables,Crítico,"Las reglas solo producen puede_solicitar = ['no']; R03 exige 'sí', que ninguna regla produce.","Cierre de mundo cerrado condicionado: si todas las reglas que concluyen puede_solicitar pudieron evaluarse y ninguna se disparó, se publica puede_solicitar ..."
3,C02 / R03,Salidas inalcanzables,Crítico,"Las reglas solo producen propiedad_solicitable = ['no']; R03 exige 'sí', que ninguna regla produce.","Cierre de mundo cerrado condicionado: si todas las reglas que concluyen propiedad_solicitable pudieron evaluarse y ninguna se disparó, se publica propiedad_..."
4,C06 / R21–R23,Salidas inalcanzables,Crítico,"Las reglas solo producen restriccion_dura_violada = ['si']; R21–R23 exigen 'no', que ninguna regla produce.","Cierre de mundo cerrado condicionado: si todas las reglas que concluyen restriccion_dura_violada pudieron evaluarse y ninguna se disparó, se publica restric..."
5,C05 / R29,Salidas inalcanzables,Crítico,"Las reglas solo producen compensacion_disponible = ['si']; R29 exige 'no', que ninguna regla produce.","Cierre de mundo cerrado condicionado: si todas las reglas que concluyen compensacion_disponible pudieron evaluarse y ninguna se disparó, se publica compensa..."
6,D01 aprobada,Salidas inalcanzables,Crítico,"Como consecuencia, nivel_compatibilidad nunca se deriva y el dictamen 'aprobada' es INALCANZABLE: con el Excel literal, SECA nunca aprueba a nadie.",Se resuelve con los cierres anteriores (sin inventar reglas nuevas).
7,R11,Valores,Alto,"'media como máximo (no puede ser alta)' es un tope escrito en lenguaje natural, no un valor.","Se extrae 'media' y nivel_solvencia usa política 'mínimo': min(alta, media) = media; min(baja, media) = baja."
8,"R16, R18, R19",Integridad,Alto,Condiciones sobre variables no declaradas: R16: 'flexible'; R18: 'diferencia_de_niveles'; R19: 'diferencia_de_niveles'.,R16 → 3 condiciones atómicas · R18/R19 → variable derivada brecha_limpieza = rango(casa) − rango(candidata).
9,"R03, R36, R37",Salidas,Alto,"Dos conclusiones en una sola celda ('estado_solicitud (+ acción …)', 'generar_explicacion + registrar_dictamen').","Se separan en una lista de conclusiones: [('estado_solicitud','pending'), ('acciones','notificar_…')]."
10,"R05, R34, R35, R36",Salidas,Alto,'accion' no está en el Diccionario y 4 reglas le asignan valores distintos: con asignación simple la última borra a las demás.,Variable 'acciones' de tipo conjunto: cada regla agrega su acción; ninguna borra.


**Lectura de la auditoría.** El Excel está bien como documento de conocimiento, pero **no es ejecutable tal cual**:

1. **Salidas binarias con una sola polaridad.** C01, C02, C05 y C06 solo se concluyen en un sentido; las reglas que esperan el otro valor nunca se activan. En cadena, eso bloquea la compatibilidad y hace **inalcanzable el dictamen "aprobada"**.
2. **Salidas que se acumulan, no se asignan.** `diferencias_blandas` (+1/+2) y `accion` necesitan acumularse; con asignación simple, la última regla borra a las anteriores.
3. **Valores en lenguaje natural.** Topes ("como máximo"), intervalos sin extremos definidos, "vacío", conjuntos de variables.

Las secciones 4 y 5 resuelven cada punto con **datos** (escalas, políticas, normalización), no con `if` en el motor.

## 4. Pesos y políticas de combinación

Cada **salida** del Excel tiene un comportamiento distinto cuando varias reglas opinan sobre ella. En la Sesión 6 vimos el problema con R09/R10 (dos prioridades distintas). Para SECA lo resolvemos declarando, por variable, una **política de combinación**:

| Tipo de salida | Variables | Política | Por qué |
|---|---|---|---|
| Booleana "basta una" | restricción dura (C06), compensación (C05) | `o` | Una sola restricción basta para violar la casa |
| Booleana "todas deben cumplirse" | puede_solicitar (C01), propiedad_solicitable (C02), expediente_completo (C03) | `y` | Basta un faltante para que sea "no" |
| Ordinal conservadora | nivel_solvencia (C04), nivel_compatibilidad (C08) | `mínimo` | Si dos reglas opinan, gana la más prudente (así funciona el tope de R11) |
| Ordinal de severidad | dictamen (D01) | `máximo` | Gana el dictamen más severo: el orden de **E02** |
| Acumulador con pesos | diferencias_blandas (C07) | `suma` | +1 / +2 de R17–R20 |
| Conjunto | acciones (A) | `conjunto` | Cada regla agrega; nadie borra |
| Valor único | condicion_aplicada, estados | `único` | Si aparecen dos valores distintos → **conflicto** registrado |

**Cierre y respaldo.** Cuando ninguna regla concluye una variable:
- si **todas** sus reglas pudieron evaluarse → se publica el valor de **cierre** (mundo cerrado: "si nada lo contradice, es así");
- si alguna quedó **indeterminada** por falta de datos → se publica el **respaldo** (casi siempre `None`, desconocido). El dictamen tiene respaldo `revision_manual`: por definición de D01, cuando el sistema no puede decidir, lo turna a una persona.

**Absorbente.** Hay valores que ningún dato faltante puede cambiar: un `sí` en restricción dura, un `baja` en solvencia o un `rechazada` en el dictamen. Esos se publican aunque falten datos.

In [17]:
POLITICAS = {
    "puede_solicitar":             {"politica": "y",        "cierre": True,  "respaldo": None,
                                    "por_que": "R01 solo define 'no'. Si R01 pudo evaluarse y no se disparó → sí (C01)."},
    "propiedad_solicitable":       {"politica": "y",        "cierre": True,  "respaldo": None,
                                    "por_que": "R02 solo define 'no'. Ídem (C02)."},
    "expediente_completo":         {"politica": "y",        "cierre": True,  "respaldo": None,
                                    "por_que": "R04 solo define 'no'. Basta un faltante."},
    "restriccion_dura_violada":    {"politica": "o",        "cierre": False, "respaldo": None,
                                    "por_que": "Basta una de R14–R16. 'no' solo si las tres pudieron evaluarse."},
    "compensacion_disponible":     {"politica": "o",        "cierre": False, "respaldo": None,
                                    "por_que": "Basta una de R12–R13."},
    "diferencias_blandas":         {"politica": "suma",     "cierre": 0,     "respaldo": None,
                                    "por_que": "R17–R20 suman puntos (+1/+2) a partir de 0."},
    "nivel_solvencia":             {"politica": "minimo",   "cierre": None,  "respaldo": None,
                                    "por_que": "Gana la regla más conservadora: R11 funciona como tope."},
    "nivel_compatibilidad":        {"politica": "minimo",   "cierre": None,  "respaldo": None,
                                    "por_que": "R21–R23 son excluyentes; 'mínimo' es la salida prudente si dejaran de serlo."},
    "dictamen":                    {"politica": "maximo",   "cierre": "revision_manual", "respaldo": "revision_manual",
                                    "por_que": "Gana el más severo (E02). Sin información suficiente → revisión manual (D01)."},
    "condicion_aplicada":          {"politica": "unico",    "cierre": "ninguna", "respaldo": None,
                                    "por_que": "R32/R33 son excluyentes; si no aplica ninguna → 'ninguna' (D02)."},
    "acciones":                    {"politica": "conjunto", "cierre": frozenset(), "respaldo": frozenset(),
                                    "por_que": "Cada regla agrega su acción."},
    "estado_solicitud":            {"politica": "unico",    "cierre": None,  "respaldo": None,
                                    "por_que": "R03 → pending. Si llega como entrada, la entrada prevalece."},
    "estado_solicitud_resultante": {"politica": "unico",    "cierre": None,  "respaldo": None,
                                    "por_que": "R37/R38 son excluyentes."},
    "decision_permitida":          {"politica": "y",        "cierre": True,  "respaldo": None,
                                    "por_que": "R39 solo define 'no permitida'."},
}
POLITICA_POR_DEFECTO = {"politica": "unico", "cierre": None, "respaldo": None,
                        "por_que": "Sin política declarada: un valor; si hay dos distintos, conflicto."}

def fmt_simple(v):
    # Ojo: {False: …}.get(0) devolvería el valor de False, porque en Python 0 == False. Por eso se compara por tipo.
    if isinstance(v, frozenset): return "∅ (conjunto vacío)"
    if v is None:                return "— (desconocido)"
    if isinstance(v, bool):      return "sí" if v else "no"
    return v

# Cómo se combinan los aportes de varias reglas sobre la misma variable
COMBINAR = {
    "y":        lambda valores, escala: all(valores),
    "o":        lambda valores, escala: any(valores),
    "suma":     lambda valores, escala: sum(valores),
    "minimo":   lambda valores, escala: min(valores, key=escala.index),
    "maximo":   lambda valores, escala: max(valores, key=escala.index),
    "conjunto": lambda valores, escala: frozenset(valores),
    "unico":    lambda valores, escala: valores[0],
}
# ¿El valor ya no puede cambiar aunque falten datos?
ABSORBENTE = {
    "y":        lambda v, escala: v is False,
    "o":        lambda v, escala: v is True,
    "suma":     lambda v, escala: False,
    "minimo":   lambda v, escala: v == escala[0],
    "maximo":   lambda v, escala: v == escala[-1],
    "conjunto": lambda v, escala: True,
    "unico":    lambda v, escala: False,
}
# Políticas que no admiten dos valores distintos (si aparecen → conflicto registrado)
EXIGE_UNICIDAD = {"unico": True}
# Tabla de publicación: (¿hubo aportes?, ¿valor certero?) → qué se publica
PUBLICACION = {
    (True,  True):  "combinado",
    (True,  False): "respaldo",
    (False, True):  "cierre",
    (False, False): "respaldo",
}

pd.DataFrame([{"variable": v, "política": p["politica"], "cierre": fmt_simple(p["cierre"]),
               "respaldo": fmt_simple(p["respaldo"]), "por qué": p["por_que"]} for v, p in POLITICAS.items()])

,variable,política,cierre,respaldo,por qué
0,puede_solicitar,y,sí,— (desconocido),R01 solo define 'no'. Si R01 pudo evaluarse y no se disparó → sí (C01).
1,propiedad_solicitable,y,sí,— (desconocido),R02 solo define 'no'. Ídem (C02).
2,expediente_completo,y,sí,— (desconocido),R04 solo define 'no'. Basta un faltante.
3,restriccion_dura_violada,o,no,— (desconocido),Basta una de R14–R16. 'no' solo si las tres pudieron evaluarse.
4,compensacion_disponible,o,no,— (desconocido),Basta una de R12–R13.
5,diferencias_blandas,suma,0,— (desconocido),R17–R20 suman puntos (+1/+2) a partir de 0.
6,nivel_solvencia,minimo,— (desconocido),— (desconocido),Gana la regla más conservadora: R11 funciona como tope.
7,nivel_compatibilidad,minimo,— (desconocido),— (desconocido),R21–R23 son excluyentes; 'mínimo' es la salida prudente si dejaran de serlo.
8,dictamen,maximo,revision_manual,revision_manual,Gana el más severo (E02). Sin información suficiente → revisión manual (D01).
9,condicion_aplicada,unico,ninguna,— (desconocido),R32/R33 son excluyentes; si no aplica ninguna → 'ninguna' (D02).


## 5. Compilación de reglas: del Excel a estructuras de Python

Seguimos el formato de la Sesión 5, extendido con operador:

```python
{"id": "R06",
 "condiciones":  [("ratio_renta_ingreso", "le", 33), ("antiguedad_laboral_meses", "ge", 6)],
 "conclusiones": [("nivel_solvencia", "alta")],
 "origen": "IA generativa", "estado": "Pendiente"}
```

El parser también es **dirigido por tablas**: operadores, valores especiales y constructores de valor son diccionarios y listas de predicados. Solo 4 reglas no se pueden interpretar automáticamente; esas van en `NORMALIZACION_V1`, cada una con su motivo, y son exactamente las que la auditoría pide corregir en el Excel v1.2.

In [18]:
class Ref:
    # Referencia a otra variable de la memoria de trabajo (comparación candidata ↔ casa)
    def __init__(self, nombre): self.nombre = nombre
    def __repr__(self): return f"Ref({self.nombre!r})"
    def __eq__(self, otro): return isinstance(otro, Ref) and otro.nombre == self.nombre
    def __hash__(self): return hash(("Ref", self.nombre))

EXISTENCIA = {"es_desconocido", "es_conocido"}
OPERADORES_ESPECIALES = {("eq", "vacio"): "es_desconocido", ("ne", "vacio"): "es_conocido"}
PATRONES_LIMPIEZA = [(r"^\s*([a-z]+)[\s_]+como[\s_]+m[aá]ximo.*$", r"\1")]   # 'media como máximo (…)' → 'media'

def limpiar(texto):
    for patron, reemplazo in PATRONES_LIMPIEZA:
        texto = re.sub(patron, reemplazo, texto, flags=re.I)
    return texto

def numeros(texto):
    return tuple(num(x) for x in re.findall(r"\d+(?:\.\d+)?", texto))

# Constructores del valor esperado: gana el primero cuyo predicado se cumple
CONSTRUCTORES_VALOR = [
    (lambda var, op, txt: op in EXISTENCIA,        lambda var, op, txt: None),
    (lambda var, op, txt: canon(txt) in VARIABLES, lambda var, op, txt: Ref(canon(txt))),
    (lambda var, op, txt: op == "entre",           lambda var, op, txt: numeros(txt)),
    (lambda var, op, txt: op in ("en", "no_en"),   lambda var, op, txt: frozenset(a_python(var, t) for t in txt.split(";"))),
    (lambda var, op, txt: True,                    lambda var, op, txt: a_python(var, limpiar(txt))),
]

def compilar_condicion(variable, op_texto, valor_texto):
    variable = canon(variable)
    op = OPERADOR_TEXTO[op_texto.strip().lower()]
    op = OPERADORES_ESPECIALES.get((op, canon(valor_texto)), op)
    construir = next(c for p, c in CONSTRUCTORES_VALOR if p(variable, op, valor_texto))
    return (variable, op, construir(variable, op, valor_texto))

def compilar_conclusiones(texto, valor):
    return [(var, a_python(var, limpiar(val))) for var, val in conclusiones_crudas({"Conclusión": texto, "Valor conclusión": valor})]

In [19]:
# Las únicas reglas que se reescriben a mano. Cada entrada es una corrección pendiente para el Excel v1.2.
NORMALIZACION_V1 = {
    "R16": {"condiciones": [("horario_actividad", "ne", Ref("casa_horario_predominante")),
                            ("horario_actividad", "ne", "flexible"),
                            ("casa_horario_predominante", "ne", "flexible")],
            "motivo": "'flexible' aparece como variable y el valor es un conjunto de variables. Se reescribe en 3 condiciones atómicas (flexible es compatible con todo, E10)."},
    "R18": {"condiciones": [("brecha_limpieza", "eq", 1)],
            "motivo": "'diferencia de niveles' no está declarada. brecha_limpieza = rango(casa_limpieza_esperada) − rango(nivel_limpieza); 'es menor a' queda implícito en brecha > 0."},
    "R19": {"condiciones": [("brecha_limpieza", "eq", 2)],
            "motivo": "Ídem R18."},
    "R39": {"conclusiones": [("decision_permitida", False)],
            "motivo": "'decisión de la anfitriona = no permitida (BAD_REQUEST)' es un error de la API. Se codifica como decision_permitida = False."},
}

# Conectores: Y → una regla con todas las condiciones · O → una regla por rama
CONECTORES = {"Y": lambda conds: [conds], "": lambda conds: [conds], "O": lambda conds: [[c] for c in conds]}
SUFIJOS = {1: [""], 2: ["a", "b"], 3: ["a", "b", "c"]}

def compilar_reglas(xl, normalizacion):
    reglas = []
    for orden, fila in enumerate(xl.to_dict("records"), start=1):
        rid, ajuste = fila["ID"], normalizacion.get(fila["ID"], {})
        conds = ajuste.get("condiciones") or [compilar_condicion(v, o, x) for v, o, x in
                                              [(fila["Condición 1"], fila["Operador"], fila["Valor 1"]),
                                               (fila["Condición 2"], fila["Operador2"], fila["Valor 2"])] if v]
        concl = ajuste.get("conclusiones") or compilar_conclusiones(fila["Conclusión"], fila["Valor conclusión"])
        ramas = CONECTORES[ajuste and "Y" or fila["Conector"]](conds)
        for sufijo, rama in zip(SUFIJOS[len(ramas)], ramas):
            reglas.append({
                "id": rid + sufijo, "fuente": rid, "orden": orden,
                "condiciones": rama, "conclusiones": concl,
                "origen": fila["Origen"], "validacion": fila["Fuente de validación"], "estado": fila["Estado"],
                "situacion_app": fila["Situación en la app"],
                "normalizada": ajuste.get("motivo", ""),
                "tope": bool(re.search(r"como m[aá]ximo", fila["Valor conclusión"], re.I)),
            })
    return reglas

REGLAS = compilar_reglas(xl_reglas, NORMALIZACION_V1)
print(f"{len(xl_reglas)} reglas en el Excel → {len(REGLAS)} reglas compiladas "
      f"({sum(r['id'] != r['fuente'] for r in REGLAS)} ramas por disyunción, {len(NORMALIZACION_V1)} normalizadas a mano)")

39 reglas en el Excel → 41 reglas compiladas (4 ramas por disyunción, 4 normalizadas a mano)


### 5.1 Variables derivadas

La fórmula de V08 está en su descripción (`V07 ÷ V06 × 100`) y R18/R19 necesitan la brecha de limpieza. Las declaramos como **datos** (función + argumentos); las funciones son genéricas y respetan lo desconocido.

In [20]:
def porcentaje(a, b):
    return None if a is None or b is None or b <= 0 else round(100 * a / b, 2)

def resta(a, b):
    return None if a is None or b is None else a - b

FUNCIONES = {
    "porcentaje":     lambda h, num_, den: porcentaje(h.get(num_), h.get(den)),
    "brecha_ordinal": lambda h, a, b: resta(CLAVES[a](h.get(a)), CLAVES[b](h.get(b))),
}
DERIVADAS = {
    "ratio_renta_ingreso": ("porcentaje",     ("renta_mensual_referencia", "ingreso_mensual_comprobable")),  # V08
    "brecha_limpieza":     ("brecha_ordinal", ("casa_limpieza_esperada", "nivel_limpieza")),              # R18/R19
}
FUNCIONES["porcentaje"]({"renta_mensual_referencia": 7500, "ingreso_mensual_comprobable": 30000}, *DERIVADAS["ratio_renta_ingreso"][1])

25.0

### 5.2 Así quedó la base compilada

Formato legible de cada regla, junto a lo que decía el Excel.

In [21]:
SIMBOLO = {"eq": "=", "ne": "≠", "lt": "<", "le": "≤", "gt": ">", "ge": "≥", "entre": "∈", "en": "∈",
           "no_en": "∉", "es_desconocido": "es desconocido", "es_conocido": "es conocido"}

def fmt(v):
    if isinstance(v, bool): return "sí" if v else "no"
    if v is None:           return "desconocido"
    if isinstance(v, Ref):  return v.nombre
    if isinstance(v, tuple): return f"({v[0]}, {v[1]}]"
    if isinstance(v, (set, frozenset)): return "{" + ", ".join(sorted(map(str, v))) + "}"
    return str(v)

def texto_condicion(c, hechos=None):
    var, op, val = c
    base = f"{var} {SIMBOLO[op]}" + ("" if op in EXISTENCIA else f" {fmt(val)}")
    return base + ("" if hechos is None else f" [{fmt(hechos.get(var))}]")

def texto_regla(r, hechos=None):
    si = " Y ".join(texto_condicion(c, hechos) for c in r["condiciones"])
    entonces = " Y ".join(f"{v} = {fmt(x)}" for v, x in r["conclusiones"])
    return f"SI {si} ENTONCES {entonces}"

excel_txt = {f["ID"]: (f"{f['Condición 1']} {f['Operador']} {f['Valor 1']}"
                       + (f" {f['Conector']} {f['Condición 2']} {f['Operador2']} {f['Valor 2']}" if f["Condición 2"] else "")
                       + f" → {f['Conclusión']} = {f['Valor conclusión']}") for f in filas}
tabla_reglas = pd.DataFrame([{"id": r["id"], "Excel": excel_txt[r["fuente"]], "Python (compilada)": texto_regla(r),
                              "origen": r["origen"], "normalizada": "✎" if r["normalizada"] else ""} for r in REGLAS])
tabla_reglas

,id,Excel,Python (compilada),origen,normalizada
0,R01,estado_verificacion_identidad es diferente de verified → puede_solicitar = no,SI estado_verificacion_identidad ≠ verified ENTONCES puede_solicitar = no,Datos / sistema existente,
1,R02,estado_propiedad es diferente de published → propiedad_solicitable = no,SI estado_propiedad ≠ published ENTONCES propiedad_solicitable = no,Datos / sistema existente,
2,R03,puede_solicitar es igual a sí Y propiedad_solicitable es igual a sí → estado_solicitud (+ acción notificar_anfitriona_nueva_solicitud) = pending,SI puede_solicitar = sí Y propiedad_solicitable = sí ENTONCES estado_solicitud = pending Y acciones = notificar_anfitriona_nueva_solicitud,Datos / sistema existente,
3,R04a,metodo_pago_validado es igual a no O ingreso_mensual_comprobable es igual a vacío → expediente_completo = no,SI metodo_pago_validado = no ENTONCES expediente_completo = no,IA generativa,
4,R04b,metodo_pago_validado es igual a no O ingreso_mensual_comprobable es igual a vacío → expediente_completo = no,SI ingreso_mensual_comprobable es desconocido ENTONCES expediente_completo = no,IA generativa,
5,R05,expediente_completo es igual a no → accion = solicitar_documentacion_faltante,SI expediente_completo = no ENTONCES acciones = solicitar_documentacion_faltante,IA generativa,
6,R06,ratio_renta_ingreso es menor o igual a 33% Y antiguedad_laboral_meses es mayor o igual a 6 → nivel_solvencia = alta,SI ratio_renta_ingreso ≤ 33 Y antiguedad_laboral_meses ≥ 6 ENTONCES nivel_solvencia = alta,IA generativa,
7,R07,ratio_renta_ingreso es menor o igual a 33% Y antiguedad_laboral_meses es menor a 6 → nivel_solvencia = media,SI ratio_renta_ingreso ≤ 33 Y antiguedad_laboral_meses < 6 ENTONCES nivel_solvencia = media,IA generativa,
8,R08,ratio_renta_ingreso entre 33% y 45% → nivel_solvencia = media,"SI ratio_renta_ingreso ∈ (33, 45] ENTONCES nivel_solvencia = media",IA generativa,
9,R09,ratio_renta_ingreso es mayor a 45% → nivel_solvencia = baja,SI ratio_renta_ingreso > 45 ENTONCES nivel_solvencia = baja,IA generativa,


In [22]:
# La regla R06 como estructura de datos (igual que en la Sesión 5)
{k: v for k, v in next(r for r in REGLAS if r["id"] == "R06").items() if k in ("id", "condiciones", "conclusiones", "origen", "estado")}

{'id': 'R06',
 'condiciones': [('ratio_renta_ingreso', 'le', 33),
  ('antiguedad_laboral_meses', 'ge', 6)],
 'conclusiones': [('nivel_solvencia', 'alta')],
 'origen': 'IA generativa',
 'estado': 'Pendiente'}

### 5.3 Los pesos del razonamiento, ya compilados

Los únicos pesos de SECA v1 son los puntos de diferencias blandas y los umbrales que los convierten en nivel de compatibilidad, más la severidad del dictamen.

In [23]:
por_id_regla = {r["id"]: r for r in REGLAS}
pesos = pd.DataFrame([{"regla": r["id"], "diferencia blanda": texto_regla(r).split(" ENTONCES ")[0][3:], "peso": x}
                      for r in REGLAS for v, x in r["conclusiones"] if v == "diferencias_blandas"])
umbrales = pd.DataFrame([{"regla": r["id"], "si": texto_regla(r).split(" ENTONCES ")[0][3:], "nivel_compatibilidad": x}
                         for r in REGLAS for v, x in r["conclusiones"] if v == "nivel_compatibilidad"])
severidad = pd.DataFrame({"dictamen": ESCALAS["dictamen"], "severidad (máximo gana)": range(len(ESCALAS["dictamen"]))})
display(pesos, umbrales, severidad)

,regla,diferencia blanda,peso
0,R17,fuma = solo_afuera Y casa_acepta_fumar = no,1
1,R18,brecha_limpieza = 1,1
2,R19,brecha_limpieza = 2,2
3,R20,frecuencia_visitas > casa_tolerancia_visitas,1


,regla,si,nivel_compatibilidad
0,R21,restriccion_dura_violada = no Y diferencias_blandas = 0,alta
1,R22,restriccion_dura_violada = no Y diferencias_blandas = 1,media
2,R23,restriccion_dura_violada = no Y diferencias_blandas ≥ 2,baja


,dictamen,severidad (máximo gana)
0,aprobada,0
1,aprobada_con_condicion,1
2,revision_manual,2
3,rechazada_para_habitacion,3


## 6. Validación de la base compilada

Antes de razonar, el motor exige que la base sea consistente. Incluye la **guarda de no discriminación**: ninguna regla puede usar variables marcadas como *Excluida* (V34 tiene_hijos, V35 edad).

In [24]:
def variables_leidas(regla):
    return ({c[0] for c in regla["condiciones"]}
            | {c[2].nombre for c in regla["condiciones"] if isinstance(c[2], Ref)})

def validar_base(reglas):
    usadas     = {v for r in reglas for v in variables_leidas(r)}
    concluidas = {v for r in reglas for v, _ in r["conclusiones"]}
    declaradas = set(VARIABLES) | set(DERIVADAS)
    acciones   = {x for r in reglas for v, x in r["conclusiones"] if v == "acciones"}
    constantes = [(r["id"], v, x) for r in reglas for v, op, x in r["condiciones"]
                  if op not in EXISTENCIA and not isinstance(x, Ref) and v in VARIABLES]
    fuera = [f"{i}: {v} {fmt(x)}" for i, v, x in constantes
             if not all(VALIDA[VARIABLES[v]["tipo"]](e, VARIABLES[v]) for e in (x if isinstance(x, (tuple, frozenset)) else [x]))]
    fuera += [f"{r['id']}: {v} = {fmt(x)}" for r in reglas for v, x in r["conclusiones"]
              if v in VARIABLES and v != "acciones" and not VALIDA[VARIABLES[v]["tipo"]](x, VARIABLES[v])]
    escalas_distintas = [f"{r['id']}: {v} vs {x.nombre}" for r in reglas for v, op, x in r["condiciones"]
                         if isinstance(x, Ref) and ESCALAS.get(v) != ESCALAS.get(x.nombre)]
    sin_escala = sorted(v for v in concluidas
                        if POLITICAS.get(v, POLITICA_POR_DEFECTO)["politica"] in ("minimo", "maximo") and v not in ESCALAS)
    topes_mal = [r["id"] for r in reglas if r["tope"]
                 and any(POLITICAS.get(v, POLITICA_POR_DEFECTO)["politica"] != "minimo" for v, _ in r["conclusiones"])]
    ids = [r["id"] for r in reglas]
    try:
        estratificar(reglas); ciclo = []
    except AssertionError as e:
        ciclo = [str(e)]
    checks = {
        "IDs únicos":                                           [i for i, n in Counter(ids).items() if n > 1],
        "Toda regla tiene condiciones y conclusiones":          [r["id"] for r in reglas if not r["condiciones"] or not r["conclusiones"]],
        "Variables de condición declaradas":                    sorted(usadas - declaradas),
        "Conclusiones declaradas":                              sorted(concluidas - declaradas),
        "Sin variables excluidas (no discriminación)":          sorted((usadas | concluidas) & EXCLUIDAS),
        "Toda conclusión tiene política de combinación":        sorted(concluidas - set(POLITICAS)),
        "Acciones existen en el Diccionario (A01–A10)":         sorted(acciones - ACCIONES_DECLARADAS),
        "Valores dentro del dominio de su variable":            fuera,
        "Comparaciones candidata ↔ casa con la misma escala":   escalas_distintas,
        "Topes ('como máximo') sobre política mínimo":          topes_mal,
        "Políticas mínimo/máximo solo sobre escalas ordinales": sin_escala,
        "Grafo de dependencias sin ciclos":                     ciclo,
    }
    tabla = pd.DataFrame([{"verificación": k, "resultado": "✓" if not v else "✗", "detalle": ", ".join(map(str, v))}
                          for k, v in checks.items()])
    return tabla, all(not v for v in checks.values())

## 7. Estratificación y flujo del razonamiento

Idea clave: **una regla solo debe leer una conclusión cuando esa conclusión ya es definitiva.** Si R32 lee `dictamen = aprobada_con_condicion` y después R31 cambia el dictamen a `revision_manual`, la condición aplicada quedaría incoherente.

Calculamos automáticamente, a partir del grafo *lee → escribe*, el **nivel** de cada regla y de cada variable:
- `nivel(regla) = 1 + max(nivel de lo que lee)` (las entradas tienen nivel 0);
- `nivel(variable) = max(nivel de las reglas que la concluyen)`.

El motor procesa los niveles en orden y **cierra** cada variable al terminar su nivel. Nadie decide el orden a mano: sale de los datos. Si el grafo tuviera un ciclo, la validación lo detecta.

In [25]:
def estratificar(reglas):
    lee      = {r["id"]: variables_leidas(r) for r in reglas}
    escribe  = {r["id"]: {v for v, _ in r["conclusiones"]} for r in reglas}
    concluidas = set().union(*escribe.values())
    nivel_var, nivel_regla, cambios, vueltas = dict.fromkeys(concluidas, 0), {}, True, 0
    while cambios:
        nivel_regla = {rid: 1 + max((nivel_var.get(v, 0) for v in lee[rid]), default=0) for rid in lee}
        nuevo = {v: max(nivel_regla[rid] for rid in escribe if v in escribe[rid]) for v in concluidas}
        cambios, nivel_var, vueltas = nuevo != nivel_var, nuevo, vueltas + 1
        assert vueltas <= len(reglas) + 1, "Hay un ciclo en el grafo de dependencias entre reglas"
    return nivel_regla, nivel_var

def construir_base(reglas):
    nivel_regla, nivel_var = estratificar(reglas)
    niveles = sorted(set(nivel_regla.values()))
    return {
        "reglas": reglas,
        "niveles": niveles,
        "nivel_regla": nivel_regla,
        "nivel_var": nivel_var,
        "reglas_por_nivel":     {n: [r for r in reglas if nivel_regla[r["id"]] == n] for n in niveles},
        "variables_por_nivel":  {n: sorted(v for v, m in nivel_var.items() if m == n) for n in niveles},
        "reglas_que_concluyen": {v: [r for r in reglas if v in {c for c, _ in r["conclusiones"]}] for v in nivel_var},
        "politicas":            {v: POLITICAS.get(v, POLITICA_POR_DEFECTO) for v in nivel_var},
        "escalas":              ESCALAS,
        "derivadas":            DERIVADAS,
    }

tabla_validacion, base_ok = validar_base(REGLAS)
assert base_ok, "La base de conocimiento no pasó la validación"
BASE = construir_base(REGLAS)
tabla_validacion

,verificación,resultado,detalle
0,IDs únicos,✓,
1,Toda regla tiene condiciones y conclusiones,✓,
2,Variables de condición declaradas,✓,
3,Conclusiones declaradas,✓,
4,Sin variables excluidas (no discriminación),✓,
5,Toda conclusión tiene política de combinación,✓,
6,Acciones existen en el Diccionario (A01–A10),✓,
7,Valores dentro del dominio de su variable,✓,
8,Comparaciones candidata ↔ casa con la misma escala,✓,
9,Topes ('como máximo') sobre política mínimo,✓,


In [26]:
flujo = pd.DataFrame(
    [{"nivel": 0, "reglas": "— (cálculos)", "variables que se cierran al terminar el nivel": ", ".join(DERIVADAS)}]
    + [{"nivel": n, "reglas": ", ".join(r["id"] for r in BASE["reglas_por_nivel"][n]),
        "variables que se cierran al terminar el nivel": ", ".join(BASE["variables_por_nivel"][n])} for n in BASE["niveles"]])
flujo

,nivel,reglas,variables que se cierran al terminar el nivel
0,0,— (cálculos),"ratio_renta_ingreso, brecha_limpieza"
1,1,"R01, R02, R04a, R04b, R06, R07, R08, R09, R10, R11, R12a, R12b, R13, R14, R15, R16, R17, R18, R19, R20, R31","compensacion_disponible, diferencias_blandas, expediente_completo, nivel_solvencia, propiedad_solicitable, puede_solicitar, restriccion_dura_violada"
2,2,"R03, R05, R21, R22, R23, R24, R28, R29","estado_solicitud, nivel_compatibilidad"
3,3,"R25, R26, R27, R30, R37, R38, R39","decision_permitida, dictamen, estado_solicitud_resultante"
4,4,"R32, R33, R34, R35, R36","acciones, condicion_aplicada"


**Cómo fluye el razonamiento de SECA:**

- **Nivel 0 · cálculos:** ratio renta/ingreso (V08) y brecha de limpieza.
- **Nivel 1 · cadenas en paralelo:** filtro de entrada (R01–R02), expediente (R04), **solvencia** (R06–R11), compensaciones (R12–R13), **restricciones duras** (R14–R16), **diferencias blandas** (R17–R20) y reportes (R31).
- **Nivel 2 · convergencia parcial:** estado de la solicitud (R03), documentación (R05), **nivel de compatibilidad** (R21–R23) y dictámenes que no necesitan compatibilidad (R24, R28, R29).
- **Nivel 3 · dictamen:** R25–R27 y R30 se suman; se cierra `dictamen` con la política de severidad (E02). También la decisión humana actual (R37–R39).
- **Nivel 4 · salidas:** condición aplicada (R32–R33) y acciones (R34–R36).

## 8. Evaluar condiciones: lógica trivalente

En la Sesión 6, `cumple()` devolvía `True` o `False`. Con datos reales aparece un tercer caso: **no lo sé**. Si la candidata no declaró si tiene mascota, R14 no es verdadera ni falsa: está **indeterminada**.

Usamos la lógica de Kleene: `False < None < True` y el **Y** de una regla es el mínimo de sus condiciones.

| | False | None | True |
|---|---|---|---|
| **False** | False | False | False |
| **None** | False | None | None |
| **True** | False | None | True |

Consecuencia útil: si la casa **acepta** mascotas, R14 es falsa aunque no sepamos si la candidata tiene mascota. El dato faltante no importa.

In [27]:
def kleene(f):
    # Con un operando desconocido, la condición es desconocida (None). Es la definición de la lógica trivalente.
    return lambda a, b, clave: None if a is None or b is None else f(a, b, clave)

OPERADORES = {
    "eq":    kleene(lambda a, b, k: k(a) == k(b)),
    "ne":    kleene(lambda a, b, k: k(a) != k(b)),
    "lt":    kleene(lambda a, b, k: k(a) <  k(b)),
    "le":    kleene(lambda a, b, k: k(a) <= k(b)),
    "gt":    kleene(lambda a, b, k: k(a) >  k(b)),
    "ge":    kleene(lambda a, b, k: k(a) >= k(b)),
    "entre": kleene(lambda a, b, k: b[0] < k(a) <= b[1]),     # intervalo semiabierto (a, b]
    "en":    kleene(lambda a, b, k: a in b),
    "no_en": kleene(lambda a, b, k: a not in b),
    "es_desconocido": lambda a, b, k: a is None,
    "es_conocido":    lambda a, b, k: a is not None,
}
RESOLVER = {Ref: lambda esperado, hechos: hechos.get(esperado.nombre)}   # por tipo del valor esperado
ORDEN_KLEENE = {False: 0, None: 1, True: 2}

def evaluar(condicion, hechos):
    variable, op, esperado = condicion
    esperado = RESOLVER.get(type(esperado), lambda e, h: e)(esperado, hechos)
    return OPERADORES[op](hechos.get(variable), esperado, CLAVES.get(variable, identidad))

def estado_regla(regla, hechos):
    # True = se cumple · False = no se cumple · None = indeterminada (faltan datos)
    return min((evaluar(c, hechos) for c in regla["condiciones"]), key=ORDEN_KLEENE.__getitem__, default=True)

Probemos con un expediente. `PERFIL_BASE` es una candidata solvente y compatible con su casa; los casos de prueba se construyen cambiando solo lo necesario.

In [28]:
PERFIL_BASE = {
    # Filtro de entrada
    "estado_verificacion_identidad": "verified",
    "estado_propiedad": "published",
    # Expediente y solvencia
    "metodo_pago_validado": True,
    "ingreso_mensual_comprobable": 30000,
    "renta_mensual_referencia": 7500,          # ratio = 25 %
    "tipo_ingreso": "nomina",
    "antiguedad_laboral_meses": 18,
    "referencias_verificadas": "si",
    "reportes_confirmados": 0,
    # Compensaciones
    "aval_disponible": False, "acepta_deposito_ampliado": False, "acepta_meses_adelantados": False,
    # Hábitos declarados por la candidata
    "horario_actividad": "early_bird", "tiene_mascota": False, "fuma": "no",
    "frecuencia_visitas": "ocasional", "nivel_limpieza": "ordenada",
    # Reglas de la casa
    "casa_acepta_mascotas": False, "casa_acepta_fumar": "no", "casa_horario_predominante": "early_bird",
    "casa_limpieza_esperada": "ordenada", "casa_tolerancia_visitas": "ocasional",
}
perfil = lambda **cambios: {**PERFIL_BASE, **cambios}

R14 = por_id_regla["R14"]
pd.DataFrame([
    {"tiene_mascota": fmt(m), "casa_acepta_mascotas": fmt(c), "R14": fmt(estado_regla(R14, perfil(tiene_mascota=m, casa_acepta_mascotas=c)))}
    for m in (True, None, False) for c in (False, True)])

,tiene_mascota,casa_acepta_mascotas,R14
0,sí,no,sí
1,sí,sí,no
2,desconocido,no,desconocido
3,desconocido,sí,no
4,no,no,no
5,no,sí,no


## 9. ¿Por qué no basta el motor de la Sesión 6?

Aplicamos a SECA el `inferir()` de la guía **con la misma estructura** (solo cambia que `cumple()` usa nuestros operadores). Tres pruebas con datos reales:

In [29]:
def cumple_v0(regla, hechos):                  # = cumple() de la Sesión 6
    for condicion in regla["condiciones"]:
        if evaluar(condicion, hechos) is not True:
            return False
    return True

def inferir_v0(hechos_iniciales, reglas):      # = inferir() de la Sesión 6
    hechos = hechos_iniciales.copy()
    reglas_disparadas = []
    cambio = True
    while cambio:
        cambio = False
        for regla in reglas:
            if regla["id"] in reglas_disparadas:
                continue
            if cumple_v0(regla, hechos):
                nuevas = [(v, x) for v, x in regla["conclusiones"] if hechos.get(v) != x]
                for variable, valor in nuevas:   # sólo agregamos conocimiento si aporta algo nuevo
                    hechos[variable] = valor
                if nuevas:
                    reglas_disparadas.append(regla["id"])
                    cambio = True
    return hechos, reglas_disparadas

def preparar(hechos):
    h = {k: v for k, v in hechos.items() if v is not None}
    for var, (f, args) in DERIVADAS.items():
        h.setdefault(var, FUNCIONES[f](h, *args))
    return h

In [30]:
# Prueba 1 · candidata ideal: solvente y 100 % compatible
h, disparadas_v0 = inferir_v0(preparar(PERFIL_BASE), REGLAS)
print("Reglas disparadas:", disparadas_v0)
print("dictamen =", h.get("dictamen"), "| nivel_compatibilidad =", h.get("nivel_compatibilidad"))
camino_aprobacion = ["R03", "R21", "R25", "R36"]
print("¿Se disparó el camino de aprobación?", {rid: rid in disparadas_v0 for rid in camino_aprobacion})

Reglas disparadas: ['R06']
dictamen = None | nivel_compatibilidad = None
¿Se disparó el camino de aprobación? {'R03': False, 'R21': False, 'R25': False, 'R36': False}


**Falla 1 — salidas inalcanzables.** La candidata ideal se queda **sin dictamen**: nadie produce `restriccion_dura_violada = no`, así que R21 nunca se activa y todo lo que depende de la compatibilidad se cae (lo que anticipó la auditoría 3.3).

In [31]:
# Prueba 2 · fuma solo afuera (+1) y recibe visitas frecuentes en casa que tolera ocasionales (+1) → deberían ser 2
h, _ = inferir_v0(preparar(perfil(fuma="solo_afuera", frecuencia_visitas="frecuente")), REGLAS)
print("v0 · diferencias_blandas =", h.get("diferencias_blandas"), "(esperado 2)")

# Y las acciones: varias reglas escriben la misma variable
h, _ = inferir_v0(preparar(perfil(tiene_mascota=True)), REGLAS)
print("v0 · acciones =", h.get("acciones"), "(esperado: sugerir_habitaciones_compatibles + generar_explicacion + registrar_dictamen + notificar)")

v0 · diferencias_blandas = 1 (esperado 2)
v0 · acciones = registrar_dictamen (esperado: sugerir_habitaciones_compatibles + generar_explicacion + registrar_dictamen + notificar)


**Falla 2 — asignación en vez de acumulación.** Dos diferencias de +1 dan 1 y la última acción borra a las demás.

In [32]:
# Prueba 3 · honorarios sin referencias completas: R06 dice 'alta', R11 dice 'media como máximo'
caso = preparar(perfil(tipo_ingreso="honorarios", referencias_verificadas="parcial"))
h_orden, _ = inferir_v0(caso, REGLAS)
h_inverso, _ = inferir_v0(caso, list(reversed(REGLAS)))
print("v0 · orden del Excel   → nivel_solvencia =", h_orden.get("nivel_solvencia"))
print("v0 · orden invertido   → nivel_solvencia =", h_inverso.get("nivel_solvencia"))

v0 · orden del Excel   → nivel_solvencia = media
v0 · orden invertido   → nivel_solvencia = alta


**Falla 3 — el resultado depende del orden de las filas del Excel.** Es exactamente el problema R09/R10 que la Sesión 6 dejó abierto: si dos reglas concluyen valores distintos, gana la última que se evaluó.

**Lo que conservamos para la v1:** `cumple()` genérico, disparo, ciclo hasta punto fijo, refracción (cada regla una vez) y traza. **Lo que añadimos:** lógica trivalente, estratos, conjunto conflicto por ciclo, políticas de combinación, cierre/respaldo y explicación.

## 10. Motor v1

El ciclo clásico de un sistema de producción (reconocer → seleccionar → actuar), procesado por niveles:

```
para cada nivel (en orden):
    mientras haya reglas activas no disparadas en el nivel:     ← RECONOCER: conjunto conflicto
        elegir una (más específica, luego orden del Excel)      ← SELECCIONAR
        disparar: registrar sus aportes                         ← ACTUAR (+ refracción)
    cerrar las variables del nivel según su política            ← COMBINAR / CIERRE / RESPALDO
```

Propiedades que buscamos (y que se verifican en la sección 13):
- **termina** siempre (cada regla se dispara a lo más una vez);
- **no depende del orden** de las filas del Excel (las políticas son conmutativas);
- **respeta lo desconocido** (un valor solo se publica si ningún dato faltante podría cambiarlo);
- **no toca las entradas** (los hechos observados prevalecen sobre lo inferido);
- **no contiene conocimiento del dominio** (ni un nombre de variable ni un valor del negocio).

In [33]:
def validar_hecho(variable, valor):
    ficha = VARIABLES.get(variable)
    if ficha is None:
        return f"'{variable}' no está declarada en el Diccionario"
    if valor is not None and not VALIDA[ficha["tipo"]](valor, ficha):
        return f"{variable} = {valor!r} no es válido para el tipo {ficha['tipo']} ({ficha.get('valores', '')})"

def memoria_de_trabajo(hechos_iniciales):
    # Aduana de entrada: tipos y dominios válidos; se descartan desconocidos (None) y variables excluidas.
    errores = [e for e in (validar_hecho(v, x) for v, x in hechos_iniciales.items()) if e]
    assert not errores, "Hechos de entrada inválidos:\n  " + "\n  ".join(errores)
    limpios = {v: x for v, x in hechos_iniciales.items() if x is not None and v not in EXCLUIDAS}
    return limpios, sorted(set(hechos_iniciales) & EXCLUIDAS)

def derivar(hechos, base):
    registro = []
    for variable, (funcion, argumentos) in base["derivadas"].items():
        calculado = FUNCIONES[funcion](hechos, *argumentos)
        registro.append({"variable": variable, "funcion": funcion, "argumentos": argumentos,
                         "calculado": calculado, "valor": hechos.setdefault(variable, calculado)})
    return registro

def conjunto_conflicto(reglas, hechos, disparadas):
    return [r for r in reglas if r["id"] not in disparadas and estado_regla(r, hechos) is True]

ESTRATEGIA = lambda r: (-len(r["condiciones"]), r["orden"])     # especificidad, luego orden del Excel

def seleccionar(conflicto):
    return min(conflicto, key=ESTRATEGIA)

def disparar(regla, aportes, disparadas):
    for variable, valor in regla["conclusiones"]:
        aportes[variable].append((valor, regla["id"]))
    disparadas.append(regla["id"])

def cerrar_variable(variable, base, hechos, aportes):
    esquema = base["politicas"][variable]
    politica, escala = esquema["politica"], base["escalas"].get(variable, [])
    valores = [valor for valor, _ in aportes[variable]]
    estados = {r["id"]: estado_regla(r, hechos) for r in base["reglas_que_concluyen"][variable]}
    completo = None not in estados.values()
    combinar = lambda: COMBINAR[politica](valores, escala)
    sin_conflicto = not EXIGE_UNICIDAD.get(politica, False) or len(set(valores)) <= 1
    certero = sin_conflicto and (completo or (bool(valores) and ABSORBENTE[politica](combinar(), escala)))
    via = PUBLICACION[(bool(valores), certero)]
    valor = {"combinado": combinar, "cierre": lambda: esquema["cierre"], "respaldo": lambda: esquema["respaldo"]}[via]()
    return {"valor_motor": valor, "valor": hechos.setdefault(variable, valor), "via": via,
            "politica": politica, "aportes": list(aportes[variable]), "completo": completo,
            "conflicto": not sin_conflicto, "indeterminadas": [rid for rid, e in estados.items() if e is None]}

def inferir(hechos_iniciales, base):
    hechos, descartados = memoria_de_trabajo(hechos_iniciales)
    entradas = dict(hechos)
    traza = {"derivaciones": derivar(hechos, base), "ciclos": [], "cierres": {}, "descartados": descartados}
    aportes, disparadas = defaultdict(list), []
    for nivel in base["niveles"]:
        while conflicto := conjunto_conflicto(base["reglas_por_nivel"][nivel], hechos, disparadas):
            regla = seleccionar(conflicto)
            disparar(regla, aportes, disparadas)
            traza["ciclos"].append({"ciclo": len(disparadas), "nivel": nivel,
                                    "conjunto conflicto": [r["id"] for r in conflicto],
                                    "disparada": regla["id"], "aporta": regla["conclusiones"]})
        for variable in base["variables_por_nivel"][nivel]:
            traza["cierres"][variable] = cerrar_variable(variable, base, hechos, aportes)
    return {"entradas": entradas, "hechos": hechos, "disparadas": disparadas, "traza": traza}

### 10.1 Vistas del resultado, explicación y registro

Estas funciones **sí** conocen el dominio (qué variables mostrar), por eso viven fuera del motor.

In [34]:
SALIDAS = ["dictamen", "condicion_aplicada", "nivel_solvencia", "nivel_compatibilidad",
           "restriccion_dura_violada", "diferencias_blandas", "compensacion_disponible",
           "expediente_completo", "acciones"]

def resumen(res):
    return {k: fmt(res["hechos"].get(k)) for k in SALIDAS}

def faltantes(res, base=None):
    base = base or BASE
    concluidas = set(base["nivel_var"])
    bloqueadas = [r for r in base["reglas"] if estado_regla(r, res["hechos"]) is None]
    desconocidas = {v for r in bloqueadas for v in variables_leidas(r)
                    if res["hechos"].get(v) is None and v not in concluidas}
    return sorted({f for v in desconocidas for f in base["derivadas"].get(v, (None, (v,)))[1]
                   if res["hechos"].get(f) is None})

def tabla_ciclos(res):
    return pd.DataFrame([{**c, "aporta": ", ".join(f"{v} = {fmt(x)}" for v, x in c["aporta"])} for c in res["traza"]["ciclos"]])

def tabla_cierres(res):
    return pd.DataFrame([{"variable": v, "valor": fmt(c["valor"]), "vía": c["via"], "política": c["politica"],
                          "aportes": ", ".join(f"{rid}→{fmt(x)}" for x, rid in c["aportes"]),
                          "¿entradas completas?": "sí" if c["completo"] else "no",
                          "indeterminadas": ", ".join(c["indeterminadas"])}
                         for v, c in res["traza"]["cierres"].items()])

SOPORTE = {"suma": lambda aportes, valor: [rid for _, rid in aportes],
           "conjunto": lambda aportes, valor: [rid for _, rid in aportes]}
POR_DEFECTO_SOPORTE = lambda aportes, valor: [rid for v, rid in aportes if v == valor]

POLITICA_TEXTO = {"y": "todas deben cumplirse", "o": "basta una regla", "suma": "suma de pesos",
                  "minimo": "gana el más conservador", "maximo": "gana el más severo",
                  "conjunto": "se acumulan", "unico": "valor único"}

def describir_via(c, valor):
    descartados = [f"{rid}→{fmt(x)}" for x, rid in c["aportes"] if x != valor and c["politica"] not in ("suma", "conjunto")]
    return {
        "combinado": f"{c['politica']}: {POLITICA_TEXTO[c['politica']]}" + (f" · descartados: {', '.join(descartados)}" if descartados else ""),
        "cierre":    "cierre: ninguna regla lo concluyó y todas pudieron evaluarse",
        "respaldo":  f"respaldo: datos insuficientes (indeterminadas: {', '.join(c['indeterminadas']) or '—'})"
                     + (f"; aportes no concluyentes: {', '.join(descartados)}" if descartados else ""),
    }[c["via"]]

def explicar(res, variable, base=None, nivel=0, vistos=None):
    base, vistos = base or BASE, vistos if vistos is not None else set()
    vistos.add(variable)
    pre, valor = "    " * nivel, res["hechos"].get(variable)
    if variable in res["entradas"]:
        return [f"{pre}• {variable} = {fmt(valor)}   ← hecho de entrada"]
    if variable in base["derivadas"]:
        f, args = base["derivadas"][variable]
        detalle = ", ".join(a + "=" + fmt(res["hechos"].get(a)) for a in args)
        return [f"{pre}• {variable} = {fmt(valor)}   ← calculado {f}({detalle})"]
    c = res["traza"]["cierres"].get(variable)
    if c is None:
        return [f"{pre}• {variable} = {fmt(valor)}   ← sin información"]
    lineas = [f"{pre}• {variable} = {fmt(valor)}   ← {describir_via(c, valor)}"]
    reglas = {r["id"]: r for r in base["reglas"]}
    soporte = SOPORTE.get(c["politica"], POR_DEFECTO_SOPORTE)(c["aportes"], valor) if c["via"] == "combinado" else []
    for rid in soporte:
        r = reglas[rid]
        lineas.append(f"{pre}    {rid} [{r['origen']} · {r['estado']}]  {texto_regla(r, res['hechos'])}")
        for v in sorted(variables_leidas(r) - vistos):
            lineas += explicar(res, v, base, nivel + 2, vistos)
    return lineas

def registrar(res, base=None, actor="SECA"):
    base = base or BASE
    serial = lambda v: sorted(v) if isinstance(v, (set, frozenset)) else v
    return {
        "motor": {"nombre": "SECA", "version": VERSION_MOTOR},
        "base_conocimiento": {"archivo": RUTA_EXCEL.name, "sha256": HASH_BC, "reglas_compiladas": len(base["reglas"])},
        "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "actor": actor,
        "entradas": {k: serial(v) for k, v in res["entradas"].items()},
        "conclusiones": {k: serial(res["hechos"].get(k)) for k in base["nivel_var"]},
        "reglas_disparadas": res["disparadas"],
        "resoluciones": [{"variable": v, "politica": c["politica"], "aportes": [[serial(x), rid] for x, rid in c["aportes"]],
                          "resultado": serial(c["valor"])}
                         for v, c in res["traza"]["cierres"].items()
                         if len({x for x, _ in c["aportes"]}) > 1 and c["politica"] not in ("suma", "conjunto")],
        "faltantes": faltantes(res, base),
        "descartados_no_discriminacion": res["traza"]["descartados"],
        "override_humano": None,
    }

## 11. Caso de punta a punta

**Expediente:** ingreso por **honorarios** con referencias solo **parciales** (ratio 25 %, 18 meses de antigüedad), **fuma solo afuera** en una casa que no permite fumar, y presenta **aval**.

Qué debería pasar según el Excel: R06 dice solvencia alta, pero R11 la topa en media; R17 suma una diferencia blanda → compatibilidad media; R27 → aprobada con condición; con aval → R32 condición = aval.

In [35]:
CASO_INTEGRADOR = perfil(tipo_ingreso="honorarios", referencias_verificadas="parcial",
                         fuma="solo_afuera", aval_disponible=True)
res = inferir(CASO_INTEGRADOR, BASE)
pd.Series(resumen(res)).to_frame("resultado")

,resultado
dictamen,aprobada_con_condicion
condicion_aplicada,aval
nivel_solvencia,media
nivel_compatibilidad,media
restriccion_dura_violada,no
diferencias_blandas,1
compensacion_disponible,sí
expediente_completo,sí
acciones,"{generar_explicacion, notificar_anfitriona_nueva_solicitud, registrar_dictamen}"


In [36]:
# Los ciclos del motor: en cada uno se recalcula el conjunto conflicto, se elige una regla y se dispara
tabla_ciclos(res)

,ciclo,nivel,conjunto conflicto,disparada,aporta
0,1,1,"[R06, R11, R12a, R17]",R06,nivel_solvencia = alta
1,2,1,"[R11, R12a, R17]",R11,nivel_solvencia = media
2,3,1,"[R12a, R17]",R17,diferencias_blandas = 1
3,4,1,[R12a],R12a,compensacion_disponible = sí
4,5,2,"[R03, R22]",R03,"estado_solicitud = pending, acciones = notificar_anfitriona_nueva_solicitud"
5,6,2,[R22],R22,nivel_compatibilidad = media
6,7,3,[R27],R27,dictamen = aprobada_con_condicion
7,8,4,"[R32, R36]",R32,condicion_aplicada = aval
8,9,4,[R36],R36,"acciones = generar_explicacion, acciones = registrar_dictamen"


In [37]:
# Cierre de cada variable al terminar su nivel: aquí se ven la combinación, el cierre y el respaldo
tabla_cierres(res)

,variable,valor,vía,política,aportes,¿entradas completas?,indeterminadas
0,compensacion_disponible,sí,combinado,o,R12a→sí,sí,
1,diferencias_blandas,1,combinado,suma,R17→1,sí,
2,expediente_completo,sí,cierre,y,,sí,
3,nivel_solvencia,media,combinado,minimo,"R06→alta, R11→media",sí,
4,propiedad_solicitable,sí,cierre,y,,sí,
5,puede_solicitar,sí,cierre,y,,sí,
6,restriccion_dura_violada,no,cierre,o,,sí,
7,estado_solicitud,pending,combinado,unico,R03→pending,sí,
8,nivel_compatibilidad,media,combinado,minimo,R22→media,sí,
9,decision_permitida,sí,cierre,y,,sí,


In [38]:
# Explicación del dictamen (A08): árbol de derivación, regla por regla, con los valores reales
print("\n".join(explicar(res, "dictamen")))
print()
print("\n".join(explicar(res, "condicion_aplicada")))

• dictamen = aprobada_con_condicion   ← maximo: gana el más severo
    R27 [Supuesto del equipo · Pendiente]  SI nivel_solvencia = media [media] Y nivel_compatibilidad ∈ {alta, media} [media] ENTONCES dictamen = aprobada_con_condicion
        • nivel_compatibilidad = media   ← minimo: gana el más conservador
            R22 [IA generativa · Pendiente]  SI restriccion_dura_violada = no [no] Y diferencias_blandas = 1 [1] ENTONCES nivel_compatibilidad = media
                • diferencias_blandas = 1   ← suma: suma de pesos
                    R17 [IA generativa · Pendiente]  SI fuma = solo_afuera [solo_afuera] Y casa_acepta_fumar = no [no] ENTONCES diferencias_blandas = 1
                        • casa_acepta_fumar = no   ← hecho de entrada
                        • fuma = solo_afuera   ← hecho de entrada
                • restriccion_dura_violada = no   ← cierre: ninguna regla lo concluyó y todas pudieron evaluarse
        • nivel_solvencia = media   ← minimo: gana el más conservador · 

In [39]:
# Registro del dictamen (A09): lo que habría que guardar en la tabla de auditoría de SheRoomie
print(json.dumps(registrar(res), ensure_ascii=False, indent=2, default=str))

{
  "motor": {
    "nombre": "SECA",
    "version": "1.0.0"
  },
  "base_conocimiento": {
    "archivo": "SECA_02_Base_Conocimiento_v1.1.xlsx",
    "sha256": "993d8cd41148",
    "reglas_compiladas": 41
  },
  "fecha_utc": "2026-09-30T21:30:15+00:00",
  "actor": "SECA",
  "entradas": {
    "estado_verificacion_identidad": "verified",
    "estado_propiedad": "published",
    "metodo_pago_validado": true,
    "ingreso_mensual_comprobable": 30000,
    "renta_mensual_referencia": 7500,
    "tipo_ingreso": "honorarios",
    "antiguedad_laboral_meses": 18,
    "referencias_verificadas": "parcial",
    "reportes_confirmados": 0,
    "aval_disponible": true,
    "acepta_deposito_ampliado": false,
    "acepta_meses_adelantados": false,
    "horario_actividad": "early_bird",
    "tiene_mascota": false,
    "fuma": "solo_afuera",
    "frecuencia_visitas": "ocasional",
    "nivel_limpieza": "ordenada",
    "casa_acepta_mascotas": false,
    "casa_acepta_fumar": "no",
    "casa_horario_predominante"

## 12. Batería de pruebas (perfiles sintéticos)

Como planteamos en la propuesta: perfiles sintéticos con salida esperada, sin datos reales de personas. Cada caso cambia solo lo necesario respecto a `PERFIL_BASE`.

- **regresión:** comportamiento que el Excel define y el motor debe respetar.
- **brecha:** el motor hace lo que dice el Excel, pero el resultado revela un hueco de conocimiento que hay que llevar a la experta.

In [40]:
BASICAS = {"notificar_anfitriona_nueva_solicitud", "generar_explicacion", "registrar_dictamen"}
acc = lambda *extra: frozenset(BASICAS | set(extra))

CASOS = [
    dict(id="T01", escenario="Solvente y 100 % compatible", hechos=perfil(), tipo="regresión", ref="R06 R21 R25",
         esperado={"dictamen": "aprobada", "condicion_aplicada": "ninguna", "acciones": acc()}),
    dict(id="T02", escenario="Solvente con mascota; la casa no acepta", hechos=perfil(tiene_mascota=True), tipo="regresión", ref="R14 R24 R34 · E01",
         esperado={"dictamen": "rechazada_para_habitacion", "acciones": acc("sugerir_habitaciones_compatibles")}),
    dict(id="T03", escenario="Ratio 40 % con aval", hechos=perfil(renta_mensual_referencia=12000, aval_disponible=True), tipo="regresión", ref="R08 R27 R32",
         esperado={"nivel_solvencia": "media", "dictamen": "aprobada_con_condicion", "condicion_aplicada": "aval"}),
    dict(id="T04", escenario="Ratio 55 % sin compensación", hechos=perfil(renta_mensual_referencia=16500), tipo="regresión", ref="R09 R29 R34",
         esperado={"nivel_solvencia": "baja", "dictamen": "rechazada_para_habitacion", "acciones": acc("sugerir_habitaciones_compatibles")}),
    dict(id="T05", escenario="Ratio 55 % que acepta depósito ampliado", hechos=perfil(renta_mensual_referencia=16500, acepta_deposito_ampliado=True), tipo="regresión", ref="R12b R28 R33",
         esperado={"dictamen": "aprobada_con_condicion", "condicion_aplicada": "deposito_ampliado"}),
    dict(id="T06", escenario="Limpieza relajada en casa muy ordenada (Δ2)", hechos=perfil(nivel_limpieza="relajada", casa_limpieza_esperada="muy_ordenada"), tipo="regresión", ref="R19 R23 R30 R35",
         esperado={"diferencias_blandas": 2, "nivel_compatibilidad": "baja", "dictamen": "revision_manual", "acciones": acc("enviar_a_revision_manual")}),
    dict(id="T07", escenario="Perfil ideal con 1 reporte confirmado", hechos=perfil(reportes_confirmados=1), tipo="regresión", ref="R25 vs R31 · E02",
         esperado={"dictamen": "revision_manual"}),
    dict(id="T08", escenario="Honorarios con referencias parciales (tope R11)", hechos=perfil(tipo_ingreso="honorarios", referencias_verificadas="parcial"), tipo="brecha", ref="R06 vs R11 · R33 · E06 E09",
         esperado={"nivel_solvencia": "media", "dictamen": "aprobada_con_condicion", "condicion_aplicada": "deposito_ampliado"}),
    dict(id="T09", escenario="Identidad pendiente de verificar", hechos=perfil(estado_verificacion_identidad="pending"), tipo="brecha", ref="R01 R03",
         esperado={"puede_solicitar": False, "estado_solicitud": None, "dictamen": "aprobada"}),
    dict(id="T10", escenario="Sin ingreso comprobable capturado", hechos=perfil(ingreso_mensual_comprobable=None), tipo="regresión", ref="R04b R05 · E04",
         esperado={"expediente_completo": False, "nivel_solvencia": None, "dictamen": "revision_manual",
                   "acciones": acc("solicitar_documentacion_faltante", "enviar_a_revision_manual")}),
    dict(id="T11", escenario="No declaró mascota; la casa no acepta", hechos=perfil(tiene_mascota=None), tipo="regresión", ref="R14 indeterminada · E17",
         esperado={"restriccion_dura_violada": None, "dictamen": "revision_manual"}),
    dict(id="T11b", escenario="No declaró mascota; la casa SÍ acepta", hechos=perfil(tiene_mascota=None, casa_acepta_mascotas=True), tipo="regresión", ref="Kleene: R14 falsa",
         esperado={"restriccion_dura_violada": False, "dictamen": "aprobada"}),
    dict(id="T12", escenario="Fuma solo afuera + visitas frecuentes", hechos=perfil(fuma="solo_afuera", frecuencia_visitas="frecuente"), tipo="regresión", ref="R17 + R20 → R23",
         esperado={"diferencias_blandas": 2, "nivel_compatibilidad": "baja", "dictamen": "revision_manual"}),
    dict(id="T13a", escenario="Ratio exactamente 33 %", hechos=perfil(renta_mensual_referencia=9900), tipo="regresión", ref="frontera R06/R08",
         esperado={"ratio_renta_ingreso": 33.0, "nivel_solvencia": "alta"}),
    dict(id="T13b", escenario="Ratio exactamente 45 %", hechos=perfil(renta_mensual_referencia=13500), tipo="regresión", ref="frontera R08/R09",
         esperado={"ratio_renta_ingreso": 45.0, "nivel_solvencia": "media"}),
    dict(id="T13c", escenario="Ratio 45.01 %", hechos=perfil(renta_mensual_referencia=13503), tipo="regresión", ref="R09",
         esperado={"nivel_solvencia": "baja"}),
    dict(id="T14", escenario="La anfitriona aprueba", hechos=perfil(decision_anfitriona="aprobar"), tipo="regresión", ref="R03 R37",
         esperado={"estado_solicitud_resultante": "confirmed", "decision_permitida": True, "acciones": acc("generar_contrato")}),
    dict(id="T15", escenario="Pago no validado, todo lo demás perfecto", hechos=perfil(metodo_pago_validado=False), tipo="brecha", ref="R04a R05 R25 · E04",
         esperado={"expediente_completo": False, "dictamen": "aprobada", "acciones": acc("solicitar_documentacion_faltante")}),
    dict(id="T16a", escenario="Madrugadora en casa nocturna", hechos=perfil(casa_horario_predominante="night_owl"), tipo="regresión", ref="R16 R24 · E10",
         esperado={"restriccion_dura_violada": True, "dictamen": "rechazada_para_habitacion"}),
    dict(id="T16b", escenario="Flexible en casa nocturna", hechos=perfil(horario_actividad="flexible", casa_horario_predominante="night_owl"), tipo="regresión", ref="R16 · E10",
         esperado={"restriccion_dura_violada": False, "dictamen": "aprobada"}),
    dict(id="T17", escenario="Sin ingreso comprobable (0) con aval", hechos=perfil(tipo_ingreso="sin_ingreso_comprobable", ingreso_mensual_comprobable=0, aval_disponible=True), tipo="regresión", ref="R10 R12a R28 R32 · E07",
         esperado={"ratio_renta_ingreso": None, "nivel_solvencia": "baja", "dictamen": "aprobada_con_condicion", "condicion_aplicada": "aval"}),
    dict(id="T18", escenario="Ratio 55 % que ofrece meses adelantados", hechos=perfil(renta_mensual_referencia=16500, acepta_meses_adelantados=True), tipo="brecha", ref="R13 R28 R33 · E09",
         esperado={"compensacion_disponible": True, "dictamen": "aprobada_con_condicion", "condicion_aplicada": "deposito_ampliado"}),
    dict(id="T19", escenario="La anfitriona rechaza", hechos=perfil(decision_anfitriona="rechazar"), tipo="regresión", ref="R38",
         esperado={"estado_solicitud_resultante": "cancelled"}),
    dict(id="T20", escenario="Solicitud ya confirmada y la anfitriona vuelve a aprobar", hechos=perfil(estado_solicitud="confirmed", decision_anfitriona="aprobar"), tipo="regresión", ref="R39 · doble rol V32",
         esperado={"estado_solicitud": "confirmed", "decision_permitida": False, "estado_solicitud_resultante": None}),
    dict(id="T21", escenario="Propiedad en borrador", hechos=perfil(estado_propiedad="draft"), tipo="regresión", ref="R02 R03",
         esperado={"propiedad_solicitable": False, "estado_solicitud": None}),
    dict(id="T22", escenario="Antigüedad de 3 meses", hechos=perfil(antiguedad_laboral_meses=3), tipo="regresión", ref="R07 R27",
         esperado={"nivel_solvencia": "media", "dictamen": "aprobada_con_condicion"}),
    dict(id="T23", escenario="Fumadora en casa que no permite fumar", hechos=perfil(fuma="si"), tipo="regresión", ref="R15 R24",
         esperado={"restriccion_dura_violada": True, "dictamen": "rechazada_para_habitacion"}),
    dict(id="T24", escenario="Limpieza relajada en casa ordenada (Δ1)", hechos=perfil(nivel_limpieza="relajada"), tipo="brecha", ref="R18 R22 R26 · E11",
         esperado={"diferencias_blandas": 1, "nivel_compatibilidad": "media", "dictamen": "aprobada"}),
    dict(id="T25", escenario="Llegan hijos y edad en la entrada", hechos=perfil(tiene_hijos=True, edad=31), tipo="regresión", ref="V34 V35 · E25",
         esperado={"dictamen": "aprobada"}),
]

# Comparación estricta: mismo tipo y mismo valor (así 0 nunca pasa por False, ni 'sí' por True)
iguales = lambda a, b: type(a) is type(b) and a == b

filas_prueba, RESULTADOS = [], {}
for caso in CASOS:
    r = inferir(caso["hechos"], BASE)
    RESULTADOS[caso["id"]] = r
    obtenido = {k: r["hechos"].get(k) for k in caso["esperado"]}
    diferencias = {k: (fmt(caso["esperado"][k]), fmt(obtenido[k])) for k in caso["esperado"] if not iguales(obtenido[k], caso["esperado"][k])}
    filas_prueba.append({"id": caso["id"], "escenario": caso["escenario"], "tipo": caso["tipo"], "reglas clave": caso["ref"],
                         "dictamen": fmt(r["hechos"].get("dictamen")), "condición": fmt(r["hechos"].get("condicion_aplicada")),
                         "ciclos": len(r["disparadas"]), "resultado": "✓" if not diferencias else f"✗ {diferencias}"})

pruebas = pd.DataFrame(filas_prueba)
print(f"{(pruebas['resultado'] == '✓').sum()} / {len(pruebas)} casos correctos")
pruebas

29 / 29 casos correctos


,id,escenario,tipo,reglas clave,dictamen,condición,ciclos,resultado
0,T01,Solvente y 100 % compatible,regresión,R06 R21 R25,aprobada,ninguna,5,✓
1,T02,Solvente con mascota; la casa no acepta,regresión,R14 R24 R34 · E01,rechazada_para_habitacion,ninguna,6,✓
2,T03,Ratio 40 % con aval,regresión,R08 R27 R32,aprobada_con_condicion,aval,7,✓
3,T04,Ratio 55 % sin compensación,regresión,R09 R29 R34,rechazada_para_habitacion,ninguna,6,✓
4,T05,Ratio 55 % que acepta depósito ampliado,regresión,R12b R28 R33,aprobada_con_condicion,deposito_ampliado,7,✓
5,T06,Limpieza relajada en casa muy ordenada (Δ2),regresión,R19 R23 R30 R35,revision_manual,ninguna,7,✓
6,T07,Perfil ideal con 1 reporte confirmado,regresión,R25 vs R31 · E02,revision_manual,ninguna,7,✓
7,T08,Honorarios con referencias parciales (tope R11),brecha,R06 vs R11 · R33 · E06 E09,aprobada_con_condicion,deposito_ampliado,7,✓
8,T09,Identidad pendiente de verificar,brecha,R01 R03,aprobada,ninguna,5,✓
9,T10,Sin ingreso comprobable capturado,regresión,R04b R05 · E04,revision_manual,ninguna,6,✓


In [41]:
assert (pruebas["resultado"] == "✓").all(), "Hay casos que no coinciden con lo esperado"
# T25: las variables excluidas se descartan antes de razonar y quedan registradas
RESULTADOS["T25"]["traza"]["descartados"], RESULTADOS["T25"]["hechos"] == RESULTADOS["T01"]["hechos"]

(['edad', 'tiene_hijos'], True)

In [42]:
# ¿Qué pasa si la app manda un dato mal codificado? La aduana lo detiene antes de razonar.
for malo in ({"fuma": True}, {"nivel_limpieza": "limpia"}, {"ratio_renta_ingreso": "33%"}, {"pet_friendly": True}):
    try:
        inferir(perfil(**malo), BASE)
    except AssertionError as e:
        print("✗", str(e).splitlines()[1].strip())

✗ fuma = True no es válido para el tipo categorico (si; no; solo_afuera)
✗ nivel_limpieza = 'limpia' no es válido para el tipo ordinal (relajada; ordenada; muy_ordenada)
✗ ratio_renta_ingreso = '33%' no es válido para el tipo numerico (numérico (%))
✗ 'pet_friendly' no está declarada en el Diccionario


## 13. Verificación del motor

Que los casos pasen no basta. Verificamos propiedades del **mecanismo**:

In [43]:
todos = list(RESULTADOS.values())
verif = []
def check(nombre, ok, detalle=""):
    verif.append({"propiedad": nombre, "resultado": "✓" if ok else "✗", "detalle": detalle})

# 1. Terminación y refracción
max_ciclos = max(len(r["disparadas"]) for r in todos)
check("Termina: ciclos ≤ nº de reglas", max_ciclos <= len(REGLAS), f"máximo {max_ciclos} ciclos con {len(REGLAS)} reglas")
check("Refracción: ninguna regla se dispara dos veces", all(len(r["disparadas"]) == len(set(r["disparadas"])) for r in todos))

# 2. Estratificación: toda regla lee solo variables ya cerradas
check("Toda regla lee variables de niveles anteriores",
      all(BASE["nivel_var"].get(v, 0) < BASE["nivel_regla"][r["id"]] for r in REGLAS for v in variables_leidas(r)))

# 3. Inmutabilidad de las entradas
caso = perfil(tipo_ingreso="honorarios"); copia = copy.deepcopy(caso); inferir(caso, BASE)
check("No modifica los hechos de entrada", caso == copia)

# 4. Determinismo y punto fijo: volver a razonar sobre la memoria final no cambia nada
check("Determinista (dos corridas iguales)", all(inferir(c["hechos"], BASE)["hechos"] == RESULTADOS[c["id"]]["hechos"] for c in CASOS))
check("Punto fijo (re-inferir la memoria final no cambia nada)",
      all(inferir(r["hechos"], BASE)["hechos"] == r["hechos"] for r in todos))

In [44]:
# 5. Independencia del orden de las reglas: barajamos el Excel 200 veces
def barajar(semilla):
    rnd = random.Random(semilla)
    reglas = [dict(r) for r in REGLAS]
    rnd.shuffle(reglas)
    for i, r in enumerate(reglas):
        r["orden"] = i
    return reglas

cambios_v1, cambios_v0 = 0, 0
for semilla in range(200):
    reglas_b = barajar(semilla)
    base_b = construir_base(reglas_b)
    cambios_v1 += sum(inferir(c["hechos"], base_b)["hechos"] != RESULTADOS[c["id"]]["hechos"] for c in CASOS)
    cambios_v0 += sum(inferir_v0(preparar(c["hechos"]), reglas_b)[0] != inferir_v0(preparar(c["hechos"]), REGLAS)[0] for c in CASOS)
total = 200 * len(CASOS)
check("Independiente del orden de las filas del Excel", cambios_v1 == 0,
      f"v1: {cambios_v1}/{total} corridas cambiaron · v0 (Sesión 6): {cambios_v0}/{total}")

In [45]:
# 6. Separación conocimiento / motor: el código del motor no menciona ningún elemento del dominio
NUCLEO = [kleene, evaluar, estado_regla, memoria_de_trabajo, derivar, conjunto_conflicto,
          seleccionar, disparar, cerrar_variable, inferir]
VOCABULARIO = (set(VARIABLES) | {t for f in CATALOGO.values() for t in f["tokens"]}
               | {v for esc in ESCALAS.values() for v in esc} | ACCIONES_DECLARADAS)

def auditar_codigo(funcion):
    arbol = ast.parse(textwrap.dedent(inspect.getsource(funcion)))
    literales = {n.value for n in ast.walk(arbol) if isinstance(n, ast.Constant) and isinstance(n.value, str)}
    return {"función": funcion.__name__,
            "literales del dominio": sorted(literales & VOCABULARIO) or "—",
            "sentencias if": sum(isinstance(n, ast.If) for n in ast.walk(arbol)),
            "expresiones if": sum(isinstance(n, ast.IfExp) for n in ast.walk(arbol))}

codigo = pd.DataFrame(map(auditar_codigo, NUCLEO))
check("El motor no contiene conocimiento del dominio", (codigo["literales del dominio"] == "—").all(),
      f"{len(NUCLEO)} funciones · {codigo['sentencias if'].sum()} sentencias if · "
      f"{codigo['expresiones if'].sum()} expresión condicional (la definición de Kleene)")
codigo

,función,literales del dominio,sentencias if,expresiones if
0,kleene,—,0,1
1,evaluar,—,0,0
2,estado_regla,—,0,0
3,memoria_de_trabajo,—,0,0
4,derivar,—,0,0
5,conjunto_conflicto,—,0,0
6,seleccionar,—,0,0
7,disparar,—,0,0
8,cerrar_variable,—,0,0
9,inferir,—,0,0


In [46]:
# 7. Guarda de no discriminación: una regla que use tiene_hijos no pasa la validación
regla_prohibida = {"id": "RX", "fuente": "RX", "orden": 99, "condiciones": [("tiene_hijos", "eq", True)],
                   "conclusiones": [("dictamen", "revision_manual")], "tope": False}
tabla_x, ok_x = validar_base(REGLAS + [regla_prohibida])
check("Rechaza reglas con variables excluidas (V34, V35)", not ok_x,
      tabla_x.loc[tabla_x["resultado"] == "✗", "detalle"].iloc[0])

# 8. Mutación del conocimiento: cambiamos el umbral 33 % → 30 % en el Excel (en memoria) sin tocar el motor
xl_mut = xl_reglas.copy()
xl_mut.loc[xl_mut["ID"].isin(["R06", "R07"]), "Valor 1"] = "30%"
xl_mut.loc[xl_mut["ID"] == "R08", "Valor 1"] = "30% y 45%"
BASE_MUT = construir_base(compilar_reglas(xl_mut, NORMALIZACION_V1))
antes = inferir(perfil(renta_mensual_referencia=9900), BASE)["hechos"]["nivel_solvencia"]
despues = inferir(perfil(renta_mensual_referencia=9900), BASE_MUT)["hechos"]["nivel_solvencia"]
check("Cambiar el Excel cambia el razonamiento sin tocar el código", (antes, despues) == ("alta", "media"),
      f"ratio 33 %: con umbral 33 → {antes} · con umbral 30 → {despues}")

# 9. Cobertura: toda regla se disparó en al menos un caso
disparadas_alguna = {rid for r in todos for rid in r["disparadas"]}
nunca = sorted({r["id"] for r in REGLAS} - disparadas_alguna)
check("Cobertura: toda regla compilada se disparó en algún caso", not nunca,
      f"{len(disparadas_alguna)}/{len(REGLAS)} reglas" + (f" · nunca: {nunca}" if nunca else ""))

verificacion = pd.DataFrame(verif)
verificacion

,propiedad,resultado,detalle
0,Termina: ciclos ≤ nº de reglas,✓,máximo 8 ciclos con 41 reglas
1,Refracción: ninguna regla se dispara dos veces,✓,
2,Toda regla lee variables de niveles anteriores,✓,
3,No modifica los hechos de entrada,✓,
4,Determinista (dos corridas iguales),✓,
5,Punto fijo (re-inferir la memoria final no cambia nada),✓,
6,Independiente del orden de las filas del Excel,✓,v1: 0/5800 corridas cambiaron · v0 (Sesión 6): 460/5800
7,El motor no contiene conocimiento del dominio,✓,10 funciones · 0 sentencias if · 1 expresión condicional (la definición de Kleene)
8,"Rechaza reglas con variables excluidas (V34, V35)",✓,tiene_hijos
9,Cambiar el Excel cambia el razonamiento sin tocar el código,✓,ratio 33 %: con umbral 33 → alta · con umbral 30 → media


In [47]:
assert (verificacion["resultado"] == "✓").all(), "Alguna propiedad del motor no se cumple"
print(f"Motor SECA v{VERSION_MOTOR}: {len(verificacion)} propiedades verificadas · {len(CASOS)} casos de prueba · base sha256 {HASH_BC}")

Motor SECA v1.0.0: 11 propiedades verificadas · 29 casos de prueba · base sha256 993d8cd41148


## 14. Brechas detectadas y preguntas para la experta

El motor aplica fielmente el Excel. Justo por eso deja al descubierto huecos que **no se deben resolver en código**, sino con la experta:

| # | Brecha | Evidencia | Pregunta (Excepción) |
|---|---|---|---|
| B1 | Un expediente incompleto (pago no validado) puede quedar **aprobado**: R05 pide documentos, pero ninguna regla de dictamen revisa `expediente_completo`. | T15 | ¿El dictamen se pausa hasta completar el expediente? **E04** |
| B2 | El filtro de entrada (identidad, propiedad publicada) **no condiciona el dictamen**: una candidata sin identidad verificada obtiene "aprobada". | T09 | ¿SECA solo corre sobre solicitudes que pasaron el filtro? **E03 / E23** |
| B3 | Si no hay aval, R33 pide **depósito ampliado aunque la candidata no lo aceptó** y aunque haya ofrecido meses adelantados. | T08, T18 | ¿Cómo se elige la condición? **E09** |
| B4 | A03–A05 (solicitar aval / depósito / meses) **nunca se disparan**: la condición se concluye, pero no la acción. | Auditoría 3.3 | Misma pregunta de **E09** |
| B5 | Solvencia alta + una diferencia blanda se **aprueba directo** (R26). | T24 | **E11** |
| B6 | Un hábito no declarado manda a **revisión manual** (comportamiento del motor por lógica trivalente). | T11 | ¿Es el comportamiento deseado? **E17** |
| B7 | Los umbrales 33 % / 45 % / 6 meses son propuesta de IA. | Pruebas T13 | **E05** |
| B8 | `cuenta_activa` (V03) no entra a ninguna regla: una cuenta suspendida puede recibir dictamen. | Auditoría 3.1 | Nueva pregunta |
| B9 | Con el Excel literal, **"aprobada" es inalcanzable** (salidas binarias con una sola polaridad). La v1 lo resuelve con cierre de mundo cerrado, pero hay que confirmar que es la lectura correcta. | Auditoría 3.3 | ¿"Si ninguna restricción dura se cumple, la casa es compatible"? |
| B10 | La anfitriona puede decidir contra el dictamen (SECA recomienda *aprobada* y ella rechaza). No está definido quién tiene la última palabra ni se guarda el motivo. | T19 | **E03 / E22** |

In [48]:
brechas = pd.DataFrame([f for f in filas_prueba if f["tipo"] == "brecha"])[["id", "escenario", "reglas clave", "dictamen", "condición"]]
brechas

,id,escenario,reglas clave,dictamen,condición
0,T08,Honorarios con referencias parciales (tope R11),R06 vs R11 · R33 · E06 E09,aprobada_con_condicion,deposito_ampliado
1,T09,Identidad pendiente de verificar,R01 R03,aprobada,ninguna
2,T15,"Pago no validado, todo lo demás perfecto",R04a R05 R25 · E04,aprobada,ninguna
3,T18,Ratio 55 % que ofrece meses adelantados,R13 R28 R33 · E09,aprobada_con_condicion,deposito_ampliado
4,T24,Limpieza relajada en casa ordenada (Δ1),R18 R22 R26 · E11,aprobada,ninguna


## 15. Ruta a la v2 (después de la entrevista con la experta)

La v2 **no debería requerir cambios en el motor** (sección 10). Lo que cambia es el conocimiento:

1. **Actualizar el Excel** con las reglas validadas (Estado = *Validado*, Fuente = *Experto del dominio*).
2. **Corregir en origen** los hallazgos de la auditoría para vaciar `NORMALIZACION_V1`:
   - R16, R18, R19 en condiciones atómicas; R39 como `decision_permitida`;
   - `accion` → `acciones`; conclusiones compuestas en filas separadas;
   - intervalos con extremos explícitos (`> 33` y `≤ 45`); `vacío` → operador *es desconocido*.
3. **Agregar columnas al Diccionario** para que las tablas de las secciones 2 y 4 salgan del Excel: `Tipo de dato`, `Escala (orden)`, `Política de combinación`, `Valor por defecto (cierre)`, `Respaldo`.
4. **Resolver las brechas B1–B9** con reglas nuevas o cambios de política.
5. **Modo validado:** filtrar `REGLAS` por `estado == "Validado"` para producción y dejar las candidatas solo en pruebas.
6. **Volver a ejecutar este notebook**: auditoría, validación, pruebas y verificación deben quedar en verde. Los casos marcados como *brecha* se convierten en *regresión* con el resultado que defina la experta.
7. **Integración con SheRoomie:** portar el motor a `packages/api/src/lib/seca/` (función pura, como `compatibility.ts`), llamarlo después de `booking.create` y persistir `registrar()` en una tabla de dictámenes.